# Pseudospace Reconstruction Method Comparison


# Constructing Pseudospace

### Shared Paths, Parameters, and Marker Panels

Defines cache locations, plotting defaults, output paths, marker panels, and per-continuum settings for the method-comparison experiments.


In [ ]:
# Standard paths, cache directories, analysis libraries, and shared constants.
from pathlib import Path
import os
import warnings

# Put package caches outside the project so plotting/numba setup does not clutter the repo.
CACHE_ROOT = Path('/tmp/pseudospace_method_comparison_cache')
CACHE_ROOT.mkdir(exist_ok=True)
(CACHE_ROOT / 'matplotlib').mkdir(exist_ok=True)
(CACHE_ROOT / 'numba').mkdir(exist_ok=True)
(CACHE_ROOT / 'xdg').mkdir(exist_ok=True)
os.environ.setdefault('NUMBA_CACHE_DIR', str(CACHE_ROOT / 'numba'))
os.environ.setdefault('MPLCONFIGDIR', str(CACHE_ROOT / 'matplotlib'))
os.environ.setdefault('XDG_CACHE_HOME', str(CACHE_ROOT / 'xdg'))

# Scientific Python stack used throughout the notebook.
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.stats import spearmanr
from IPython.display import display

sc.settings.verbosity = 2
sns.set_theme(style='whitegrid', context='notebook')
warnings.filterwarnings('ignore', category=FutureWarning)

# Project paths and output locations.
PROJECT_DIR = Path.cwd().resolve()
DATA_DIR = PROJECT_DIR / 'data'
HARMONY_PATH = DATA_DIR / 'all_tubules_harmony_ortholog_space_caleb.h5ad'
OUTPUT_DIR = DATA_DIR / 'pseudspace_analysis_caleb'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Local R library for optional Slingshot and Monocle3 dependencies.
R_LIB_DIR = PROJECT_DIR / 'r_libs'
R_LIB_DIR.mkdir(exist_ok=True)
os.environ.setdefault('R_LIBS_USER', str(R_LIB_DIR))

# Shared analysis parameters and helpers.
RANDOM_STATE = 0
MAX_R_METHOD_CELLS = None  # Use all tubules for R-based methods.
N_NEIGHBORS = 30
N_PCS_FALLBACK = 30
N_DIFFMAP_COMPONENTS_TO_TEST = 8
BROAD_TUBULE_LEIDEN_RESOLUTION = 1.0
CONTINUA_TO_RUN = ['TOTAL', 'PT', 'TAL', 'DCT']

print(f'Harmony input: {HARMONY_PATH}')
print(f'Output directory: {OUTPUT_DIR}')


# Marker panels used to assign each tubule to a broad nephron family.
BROAD_SEGMENT_MARKERS = {
    'PT': ['Slc34a1', 'Lrp2', 'Slc5a12', 'Slc13a3', 'Slc22a6', 'Slc22a7', 'Aqp1'],
    'TL': ['Aqp1', 'Cldn4', 'Id1', 'Jag1', 'Vcam1', 'Tacstd2', 'Akr1b1'],
    'TAL': ['Umod', 'Slc12a1', 'Cldn10', 'Egf', 'Bcl6'],
    'DCT': ['Slc12a3', 'Calb1', 'Pvalb', 'Hsd11b2'],
    'CNT_CD': ['Calb1', 'Aqp2', 'Aqp3', 'Itpkb', 'Fxyd4', 'Slc4a1', 'Slc26a4', 'Atp6v1b1'],
}

# Per-continuum settings: subset, expected order, marker panels, and plotting colors.
CONTINUUM_CONFIGS = {
    'TOTAL': {
        'prefix': 'total',
        'subset_family': None,
        'segment_order': ['PT', 'TL', 'TAL', 'DCT', 'CNT_CD'],
        'segment_markers': BROAD_SEGMENT_MARKERS,
        'position_markers': {
            'early': ['Slc34a1', 'Lrp2', 'Slc5a2', 'Slc5a12'],
            'late': ['Aqp2', 'Aqp3', 'Slc4a1', 'Slc26a4', 'Atp6v1b1'],
        },
        'palette': {'PT': '#4C78A8', 'TL': '#B279A2', 'TAL': '#F58518', 'DCT': '#54A24B', 'CNT_CD': '#E45756'},
    },
    'PT': {
        'prefix': 'pt',
        'subset_family': 'PT',
        'segment_order': ['PT_S1', 'PT_S2', 'PT_S3'],
        'segment_markers': {
            'PT_S1': ['Slc5a2', 'Slc5a12', "Slc5a1", "Cubn", "Gatm"], # caleb adding 3 genes 
            'PT_S2': ['Slc22a8', 'Slc22a6', 'Slc13a3', 'Slc22a2', 'Slc34a1', "Slc22a30", "Cyp2e1", "Atp11a"], # caleb adding 3 genes
            'PT_S3': ['Slc7a13', 'Slc5a8', 'Slc38a3', 'Hao2', 'Pck1', 'Acsm3', "Slc7a12", "Miox", "Kap"], # caleb adding 3 genes
        },
        'position_markers': {
            'early': ['Slc5a2', 'Slc5a12', 'Slc22a8'],
            'late': ['Slc7a13', 'Slc5a8', 'Slc38a3', 'Hao2', 'Pck1', 'Acsm3'],
        },
        'palette': {'PT_S1': '#4C78A8', 'PT_S2': '#F58518', 'PT_S3': '#54A24B'},
    },
    'TAL': {
        'prefix': 'tal',
        'subset_family': 'TAL',
        'segment_order': ['M_TAL', 'C_TAL'],
        'segment_markers': {
            'M_TAL': ['Umod', 'Slc12a1', 'Kcnj1', 'Egf'], # caleb doesnt see much UMOD expression
            'C_TAL': ['Cldn10', 'Bcl6', 'Nos1'],
        },
        'position_markers': {
            'early': ['Slc12a1', 'Kcnj1', 'Egf'],
            'late': ['Nos1', 'Bcl6'],
        },
        'palette': {'M_TAL': '#4C78A8', 'C_TAL': '#F58518'},
    },
    'DCT': {
        'prefix': 'dct',
        'subset_family': 'DCT',
        'segment_order': ['DCT1', 'DCT2'],
        'segment_markers': {
            'DCT1': ['Slc12a3', 'Pvalb'],
            'DCT2': ['Hsd11b2', 'Scnn1g', 'Calb1'],
        },
        'position_markers': {
            'early': ['Slc12a3', 'Pvalb'],
            'late': ['Hsd11b2', 'Scnn1g'],
        },
        'palette': {'DCT1': '#4C78A8', 'DCT2': '#F58518'},
    },
}

# One-time preprocessing parameters.
MIN_GENE_TUBULE_FRACTION = 0.05
MIN_GENE_TOTAL_COUNTS = 20
NORMALIZE_TARGET_SUM = 1e4
N_HVGS = 3000
REMOVE_MITO_RIBO_STRESS_GENES = False
EXCLUDE_VALIDATION_MARKERS = False

STRESS_GENE_PREFIXES = ('FOS', 'JUN', 'HSP', 'DNAJ')

display(pd.DataFrame({'continuum': list(CONTINUUM_CONFIGS.keys()), 'output_dir': [str(OUTPUT_DIR)] * len(CONTINUUM_CONFIGS)}))

## Filtering and Normalization


> **Filtering and normalization notes**
>
> The notebook loads the harmonized all-tubules AnnData object once, preserves the raw-count matrix in `layers['counts']`, and uses that same filtered matrix for all pseudotime methods.
>
> Genes are kept if they are expressed in at least `MIN_GENE_TUBULE_FRACTION` of tubules, currently 5%, and have at least `MIN_GENE_TOTAL_COUNTS` total counts, currently 20. The retained count matrix is then normalized with `sc.pp.normalize_total(target_sum=1e4)` and log-transformed with `sc.pp.log1p`; the result is stored in `layers['lognorm']`.
>
> Feature selection runs HVG selection on the tubule-level log-normalized matrix with up to `N_HVGS` genes, currently 3000. Validation markers are retained by default even if they are not HVGs. Optional switches can remove mitochondrial/ribosomal/stress genes or exclude validation markers for sensitivity analysis.


### Load, Filter, and Normalize Harmonized Matrix

Loads the harmonized tubule object, preserves raw counts, applies the expression-count gene filter, and creates log-normalized expression used by all methods.


In [ ]:
# Load once, preserve raw counts, filter genes, normalize/log-transform, and prepare continua.
adata_all = sc.read_h5ad(HARMONY_PATH)
print(f'Loaded harmonized object: {adata_all.n_obs:,} tubules x {adata_all.n_vars:,} genes')

if 'counts' in adata_all.layers:
    counts = adata_all.layers['counts'].copy()
else:
    counts = adata_all.X.copy()
    print('No counts layer found; using current X as the raw-count source')
adata_all.layers['counts'] = counts.copy()
adata_all.uns['raw_counts_layer'] = 'counts'

counts_matrix = counts if sparse.issparse(counts) else np.asarray(counts)
expressed = np.asarray((counts_matrix > 0).sum(axis=0)).ravel()
total_counts = np.asarray(counts_matrix.sum(axis=0)).ravel()
min_cells = max(1, int(np.ceil(MIN_GENE_TUBULE_FRACTION * adata_all.n_obs)))
gene_keep = (expressed >= min_cells) & (total_counts >= MIN_GENE_TOTAL_COUNTS)
adata_all.var['n_tubules_expressed'] = expressed
adata_all.var['total_counts'] = total_counts
adata_all.var['passes_expression_count_filter'] = gene_keep
print(f'Gene filter keeps {int(gene_keep.sum()):,}/{adata_all.n_vars:,} genes '
      f'(expressed in >= {MIN_GENE_TUBULE_FRACTION:.0%} tubules and total counts >= {MIN_GENE_TOTAL_COUNTS})')
adata_all = adata_all[:, gene_keep].copy()

# Normalize every method from the same filtered raw-count matrix.
# The loaded object already has log1p metadata from notebook 1, so remove it after
# replacing X with counts; otherwise scanpy warns even though X has been reset.
adata_all.X = adata_all.layers['counts'].copy()
adata_all.uns.pop('log1p', None)
sc.pp.normalize_total(adata_all, target_sum=NORMALIZE_TARGET_SUM)
sc.pp.log1p(adata_all)
adata_all.layers['lognorm'] = adata_all.X.copy()
adata_all.uns['normalization'] = {
    'method': 'normalize_total_log1p',
    'target_sum': NORMALIZE_TARGET_SUM,
    'source_layer': 'counts',
}
print("Normalized from layers['counts']; stored log-normalized values in layers['lognorm'].")

# Preserve the full expression-filtered matrix for final saved AnnData outputs.
adata_all_expression_filtered = adata_all.copy()
display(pd.DataFrame([{'tubules': adata_all.n_obs, 'genes_after_filter': adata_all.n_vars}]))

### Build Gene Lookup

Builds a case-insensitive gene lookup, including available alias columns, so marker panels resolve consistently across methods.


In [ ]:
# Build a case-insensitive gene lookup that also checks common gene-symbol alias columns.
def build_gene_lookup(adata):
    var_names = pd.Index(adata.var_names.astype(str))
    lookup = {name.upper(): name for name in var_names}
    for alias_col in ['gene_symbol', 'gene_symbols', 'symbol', 'gene_name', 'feature_name', 'features']:
        if alias_col in adata.var.columns:
            for idx_name, alias in zip(var_names, adata.var[alias_col].astype(str)):
                alias_upper = alias.upper()
                if alias_upper and alias_upper != 'NAN' and alias_upper not in lookup:
                    lookup[alias_upper] = idx_name
    return lookup

gene_lookup = build_gene_lookup(adata_all)
validation_markers = sorted(set(
    marker
    for config in CONTINUUM_CONFIGS.values()
    for marker_sets in [config['segment_markers'], config['position_markers']]
    for genes in marker_sets.values()
    for marker in genes
))
validation_genes = {gene_lookup[g.upper()] for g in validation_markers if g.upper() in gene_lookup}

if REMOVE_MITO_RIBO_STRESS_GENES:
    upper_names = pd.Index(adata_all.var_names.astype(str).str.upper())
    optional_exclude = (
        upper_names.str.startswith(('MT-', 'MT.'))
        | upper_names.str.startswith(('RPS', 'RPL'))
        | upper_names.map(lambda name: name.startswith(STRESS_GENE_PREFIXES))
    ).to_numpy(dtype=bool)
else:
    optional_exclude = np.zeros(adata_all.n_vars, dtype=bool)
adata_all.var['excluded_optional_gene'] = optional_exclude
if optional_exclude.any():
    adata_all = adata_all[:, ~optional_exclude].copy()
    gene_lookup = build_gene_lookup(adata_all)
    validation_genes = {gene_lookup[g.upper()] for g in validation_markers if g.upper() in gene_lookup}
print(f'Gene lookup contains {len(gene_lookup):,} symbols and aliases.')

### Select Pseudospace Features and Broad Families

Selects the high-variance feature matrix used for trajectory reconstruction and assigns broad nephron-family labels when needed.


In [ ]:
# Select HVGs on tubule-level expression, retaining validation markers unless sensitivity mode excludes them.
sc.pp.highly_variable_genes(adata_all, n_top_genes=min(N_HVGS, adata_all.n_vars), flavor='seurat')
validation_mask = adata_all.var_names.isin(validation_genes)
feature_keep = adata_all.var['highly_variable'].to_numpy(dtype=bool)
if EXCLUDE_VALIDATION_MARKERS:
    feature_keep = feature_keep & ~validation_mask
else:
    feature_keep = feature_keep | validation_mask
adata_all.var['selected_for_pseudotime'] = feature_keep
print(f'Feature selection keeps {int(feature_keep.sum()):,}/{adata_all.n_vars:,} genes '
      f'({int(adata_all.var["highly_variable"].sum()):,} HVGs; '
      f'{int(validation_mask.sum()):,} validation markers available)')
adata_all = adata_all[:, feature_keep].copy()
gene_lookup = build_gene_lookup(adata_all)

# Assign broad tubule families by Leiden cluster, using broad marker scores to label each cluster.
family_col = 'broad_tubule_marker_call'
score_cols = []
for segment, marker_genes in BROAD_SEGMENT_MARKERS.items():
    genes = [gene_lookup[g.upper()] for g in marker_genes if g.upper() in gene_lookup]
    if not genes:
        continue
    score_col = f'{segment}_broad_score'
    sc.tl.score_genes(adata_all, genes, score_name=score_col, use_raw=False)
    score_cols.append(score_col)
if not score_cols:
    raise ValueError('No broad family markers were found in the filtered matrix')

broad_rep = 'X_harmony' if 'X_harmony' in adata_all.obsm else 'X_pca'
broad_neighbors_key = 'broad_tubule_marker_neighbors'
broad_cluster_col = 'broad_tubule_marker_leiden'
sc.pp.neighbors(
    adata_all,
    n_neighbors=N_NEIGHBORS,
    use_rep=broad_rep,
    key_added=broad_neighbors_key,
    random_state=RANDOM_STATE,
)
sc.tl.leiden(
    adata_all,
    resolution=BROAD_TUBULE_LEIDEN_RESOLUTION,
    neighbors_key=broad_neighbors_key,
    key_added=broad_cluster_col,
    random_state=RANDOM_STATE,
)

scores = adata_all.obs[score_cols].astype(float)
cluster_scores = scores.assign(_leiden=adata_all.obs[broad_cluster_col].astype(str).to_numpy()).groupby('_leiden').mean()
cluster_calls = cluster_scores.idxmax(axis=1).str.replace('_broad_score', '', regex=False)
calls = adata_all.obs[broad_cluster_col].astype(str).map(cluster_calls)
adata_all.obs[family_col] = pd.Categorical(calls, categories=list(BROAD_SEGMENT_MARKERS), ordered=True)
cluster_call_summary = (
    pd.DataFrame({
        broad_cluster_col: cluster_calls.index,
        family_col: cluster_calls.values,
        'n_tubules': adata_all.obs[broad_cluster_col].astype(str).value_counts().reindex(cluster_calls.index).astype(int).values,
    })
    .sort_values([family_col, broad_cluster_col])
    .reset_index(drop=True)
)

display(adata_all.obs[family_col].value_counts(dropna=False))
display(cluster_call_summary)

# Plot the Leiden clusters and broad labels on the existing harmonized UMAP.
if 'X_umap' in adata_all.obsm:
    umap = adata_all.obsm['X_umap']

    fig, axes = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True)
    leiden_codes = adata_all.obs[broad_cluster_col].astype('category').cat.codes.to_numpy()
    scatter = axes[0].scatter(
        umap[:, 0], umap[:, 1], c=leiden_codes, cmap='tab20', s=3, alpha=0.75, linewidths=0
    )
    axes[0].set_title(f'Leiden clusters used for {family_col}')
    axes[0].set_xlabel('UMAP1')
    axes[0].set_ylabel('UMAP2')
    axes[0].set_xticks([])
    axes[0].set_yticks([])
    cbar = fig.colorbar(scatter, ax=axes[0], fraction=0.046, pad=0.04)
    cbar.set_label(broad_cluster_col)

    broad_palette = CONTINUUM_CONFIGS['TOTAL']['palette']
    for label in BROAD_SEGMENT_MARKERS:
        mask = adata_all.obs[family_col].astype(str).eq(label).to_numpy()
        axes[1].scatter(
            umap[mask, 0], umap[mask, 1], s=3, alpha=0.75, linewidths=0,
            color=broad_palette[label], label=label
        )
    axes[1].set_title(f'Cluster-derived {family_col}')
    axes[1].set_xlabel('UMAP1')
    axes[1].set_ylabel('UMAP2')
    axes[1].set_xticks([])
    axes[1].set_yticks([])
    axes[1].legend(markerscale=4, frameon=False, loc='center left', bbox_to_anchor=(1.02, 0.5))
    fig.savefig(OUTPUT_DIR / 'broad_tubule_leiden_cluster_umap.png', dpi=220, bbox_inches='tight')
    plt.show()

    fig, axes = plt.subplots(1, len(score_cols), figsize=(3.4 * len(score_cols), 3.5), constrained_layout=True)
    axes = np.atleast_1d(axes)
    for ax, score_col in zip(axes, score_cols):
        values = adata_all.obs[score_col].to_numpy(dtype=float)
        vmax = np.nanpercentile(values, 99)
        vmin = np.nanpercentile(values, 1)
        scatter = ax.scatter(
            umap[:, 0], umap[:, 1], c=values, cmap='viridis', s=3, alpha=0.75,
            linewidths=0, vmin=vmin, vmax=vmax
        )
        ax.set_title(score_col.replace('_broad_score', ''))
        ax.set_xlabel('UMAP1')
        ax.set_ylabel('UMAP2')
        ax.set_xticks([])
        ax.set_yticks([])
        fig.colorbar(scatter, ax=ax, fraction=0.046, pad=0.04)
    fig.suptitle('Broad tubule marker scores used to label Leiden clusters', y=1.03)
    fig.savefig(OUTPUT_DIR / 'broad_tubule_marker_score_umap.png', dpi=220, bbox_inches='tight')
    plt.show()
else:
    print('X_umap is not available; skipping broad tubule UMAP diagnostics.')

### Broad Leiden Resolution Comparison

Runs several Leiden resolutions on the same broad-tubule neighbor graph and displays each clustering on the saved UMAP. Each resolution is validated against the strongest per-tubule broad marker score and with clustering metrics calculated in Harmony space. The resolution with the lowest validation mean rank is automatically rerun as the production `broad_tubule_marker_leiden` assignment before downstream continuum objects are prepared.

**Metrics and interpretation**

- **High-margin agreement**: fraction of confidently scored tubules whose individual marker-score winner matches their cluster's majority identity. Higher is better; `>0.90` is strong and `1.0` is perfect.
- **Purity**: fraction of all tubules matching their cluster's majority marker-score identity. Higher is better, but it can increase with unnecessary over-clustering; `>0.80` is generally strong.
- **NMI (normalized mutual information)**: shared information between Leiden clusters and individual score calls, adjusted to a `0-1` scale. Higher is better; `0` means unrelated and `1` means identical assignments. Values around `0.3-0.5` indicate moderate agreement.
- **ARI (adjusted Rand index)**: pairwise agreement with individual score calls, corrected for chance. Higher is better; `0` is chance-level, `1` is perfect, and negative values are worse than chance. Values above `0.5` are commonly considered substantial.
- **Silhouette score**: separation and compactness of clusters in Harmony space, calculated on the same 10,000-tubule sample for every resolution. Higher is better; range `-1` to `1`. Above `0.5` is strong, `0.25-0.5` is moderate, and near `0` indicates overlapping clusters.
- **Calinski-Harabasz score**: between-cluster separation relative to within-cluster dispersion in Harmony space. Higher is better; it has no universal cutoff and should only be compared across these resolutions.
- **Davies-Bouldin score**: average similarity between each cluster and its most similar alternative in Harmony space. Lower is better; `0` is ideal and values below about `1` usually indicate well-separated clusters.
- **Validation mean rank**: average rank across purity, high-margin agreement, NMI, ARI, silhouette, Calinski-Harabasz, and Davies-Bouldin. Lower is better. It is a comparison aid, not an absolute biological score.

No single metric determines the correct biological resolution. Prefer a resolution with good marker agreement and embedding separation without creating many small or redundant clusters.

In [ ]:
BROAD_LEIDEN_RESOLUTIONS_TO_COMPARE = [0.2, 0.3, 0.4, 0.5, 0.6, 0.8, 1.0]
from sklearn.metrics import adjusted_rand_score, calinski_harabasz_score, davies_bouldin_score, normalized_mutual_info_score, silhouette_score

individual_score_calls = scores.idxmax(axis=1).str.replace('_broad_score', '', regex=False)
sorted_score_values = np.sort(scores.to_numpy(dtype=float), axis=1)
individual_score_margin = sorted_score_values[:, -1] - sorted_score_values[:, -2]
high_margin_mask = individual_score_margin >= np.nanmedian(individual_score_margin)

validation_rep = np.asarray(adata_all.obsm[broad_rep], dtype=float)
rng = np.random.default_rng(RANDOM_STATE)
metric_sample_size = min(10000, adata_all.n_obs)
metric_sample_idx = np.sort(rng.choice(adata_all.n_obs, size=metric_sample_size, replace=False))
resolution_rows = []

for resolution in BROAD_LEIDEN_RESOLUTIONS_TO_COMPARE:
    resolution_key = str(resolution).replace('.', '_')
    diagnostic_col = f'broad_tubule_leiden_r{resolution_key}'
    sc.tl.leiden(
        adata_all,
        resolution=resolution,
        neighbors_key=broad_neighbors_key,
        key_added=diagnostic_col,
        random_state=RANDOM_STATE,
    )
    cluster_labels = adata_all.obs[diagnostic_col].astype(str)
    cluster_to_score_call = pd.crosstab(cluster_labels, individual_score_calls).idxmax(axis=1)
    cluster_majority_calls = cluster_labels.map(cluster_to_score_call)
    sampled_labels = cluster_labels.iloc[metric_sample_idx].to_numpy()
    sampled_rep = validation_rep[metric_sample_idx]
    resolution_rows.append({
        'resolution': resolution,
        'obs_column': diagnostic_col,
        'n_clusters': int(cluster_labels.nunique()),
        'smallest_cluster': int(cluster_labels.value_counts().min()),
        'largest_cluster': int(cluster_labels.value_counts().max()),
        'individual_score_purity': float(cluster_majority_calls.eq(individual_score_calls).mean()),
        'high_margin_agreement': float(cluster_majority_calls[high_margin_mask].eq(individual_score_calls[high_margin_mask]).mean()),
        'nmi_individual_score': float(normalized_mutual_info_score(individual_score_calls, cluster_labels)),
        'ari_individual_score': float(adjusted_rand_score(individual_score_calls, cluster_labels)),
        'silhouette_harmony': float(silhouette_score(sampled_rep, sampled_labels, metric='euclidean')),
        'calinski_harabasz': float(calinski_harabasz_score(sampled_rep, sampled_labels)),
        'davies_bouldin': float(davies_bouldin_score(sampled_rep, sampled_labels)),
    })

resolution_summary = pd.DataFrame(resolution_rows)
resolution_summary['validation_mean_rank'] = pd.concat([
    resolution_summary['individual_score_purity'].rank(ascending=False),
    resolution_summary['high_margin_agreement'].rank(ascending=False),
    resolution_summary['nmi_individual_score'].rank(ascending=False),
    resolution_summary['ari_individual_score'].rank(ascending=False),
    resolution_summary['silhouette_harmony'].rank(ascending=False),
    resolution_summary['calinski_harabasz'].rank(ascending=False),
    resolution_summary['davies_bouldin'].rank(ascending=True),
], axis=1).mean(axis=1)
resolution_summary = resolution_summary.sort_values('validation_mean_rank').reset_index(drop=True)
selected_resolution = float(
    resolution_summary.sort_values(['validation_mean_rank', 'resolution']).iloc[0]['resolution']
)
BROAD_TUBULE_LEIDEN_RESOLUTION = selected_resolution
resolution_summary.to_csv(OUTPUT_DIR / 'broad_tubule_leiden_resolution_metrics.csv', index=False)
display(
    resolution_summary[[
        'resolution',
        'n_clusters',
        'high_margin_agreement',
        'nmi_individual_score',
        'ari_individual_score',
        'silhouette_harmony',
        'validation_mean_rank',
    ]].round(3)
)
print(f'Automatically selected broad-tubule Leiden resolution: {selected_resolution:g}')

# Replace the provisional production clustering and broad-family calls with the selected resolution.
sc.tl.leiden(
    adata_all,
    resolution=selected_resolution,
    neighbors_key=broad_neighbors_key,
    key_added=broad_cluster_col,
    random_state=RANDOM_STATE,
)
scores = adata_all.obs[score_cols].astype(float)
cluster_scores = scores.assign(_leiden=adata_all.obs[broad_cluster_col].astype(str).to_numpy()).groupby('_leiden').mean()
cluster_calls = cluster_scores.idxmax(axis=1).str.replace('_broad_score', '', regex=False)
calls = adata_all.obs[broad_cluster_col].astype(str).map(cluster_calls)
adata_all.obs[family_col] = pd.Categorical(calls, categories=list(BROAD_SEGMENT_MARKERS), ordered=True)
cluster_call_summary = (
    pd.DataFrame({
        broad_cluster_col: cluster_calls.index,
        family_col: cluster_calls.values,
        'n_tubules': adata_all.obs[broad_cluster_col].astype(str).value_counts().reindex(cluster_calls.index).astype(int).values,
    })
    .sort_values([family_col, broad_cluster_col])
    .reset_index(drop=True)
)
display(cluster_call_summary)

if 'X_umap' in adata_all.obsm:
    umap = adata_all.obsm['X_umap']
    n_cols = 4
    n_rows = int(np.ceil(len(resolution_summary) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.4 * n_cols, 4.1 * n_rows), constrained_layout=True)
    axes = np.asarray(axes).reshape(-1)
    categorical_cmap = plt.get_cmap('tab20')

    for ax, row in zip(axes, resolution_summary.itertuples(index=False)):
        labels = adata_all.obs[row.obs_column].astype('category')
        label_strings = labels.astype(str)
        codes = labels.cat.codes.to_numpy()
        ax.scatter(
            umap[:, 0], umap[:, 1], c=codes, cmap=categorical_cmap,
            s=3, alpha=0.8, linewidths=0, rasterized=True,
            vmin=-0.5, vmax=max(19.5, row.n_clusters - 0.5),
        )
        for cluster_label in labels.cat.categories:
            cluster_mask = label_strings.eq(str(cluster_label)).to_numpy()
            if cluster_mask.sum() == 0:
                continue
            center = np.median(umap[cluster_mask], axis=0)
            ax.text(
                center[0], center[1], str(cluster_label),
                ha='center', va='center', fontsize=8, fontweight='bold',
                color='black',
                bbox={'boxstyle': 'round,pad=0.15', 'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.72},
            )
        production_note = ' (selected production)' if np.isclose(row.resolution, selected_resolution) else ''
        ax.set_title(f'Resolution {row.resolution:g}: {row.n_clusters} clusters{production_note}')
        ax.set_xlabel('UMAP1')
        ax.set_ylabel('UMAP2')
        ax.set_xticks([])
        ax.set_yticks([])

    for ax in axes[len(resolution_summary):]:
        ax.set_visible(False)

    fig.suptitle('Broad-tubule Leiden resolution comparison on the same neighbor graph', fontsize=16, fontweight='bold')
    fig.savefig(OUTPUT_DIR / 'broad_tubule_leiden_resolution_comparison.png', dpi=220, bbox_inches='tight')
    plt.show()

    fig, ax = plt.subplots(figsize=(7.2, 5.8), constrained_layout=True)
    individual_palette = CONTINUUM_CONFIGS['TOTAL']['palette']
    for label in BROAD_SEGMENT_MARKERS:
        mask = individual_score_calls.eq(label).to_numpy()
        ax.scatter(
            umap[mask, 0], umap[mask, 1], s=3, alpha=0.75, linewidths=0,
            color=individual_palette[label], label=f'{label} ({mask.sum():,})', rasterized=True,
        )
    ax.set_title('Individual tubule call from strongest broad marker score')
    ax.set_xlabel('UMAP1')
    ax.set_ylabel('UMAP2')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.legend(frameon=False, markerscale=3, fontsize=8, loc='center left', bbox_to_anchor=(1.01, 0.5))
    fig.savefig(OUTPUT_DIR / 'broad_tubule_individual_score_calls_umap.png', dpi=220, bbox_inches='tight')
    plt.show()
else:
    print('X_umap is not available; skipping Leiden resolution comparison plot.')


### Prepare Continuum-Specific AnnData Objects

Creates one filtered AnnData object per continuum and records marker availability for roots, termini, segments, and validation panels.


In [ ]:
# Create one filtered AnnData object per continuum for all downstream pseudotime methods.
preprocessed_loaded = {}
for continuum in CONTINUA_TO_RUN:
    print('\n' + '=' * 80)
    print(f'Preparing {continuum}')
    config = CONTINUUM_CONFIGS[continuum]
    prefix = config['prefix']

    if config['subset_family'] is None:
        adata = adata_all.copy()
    else:
        mask = adata_all.obs[family_col].astype(str) == config['subset_family']
        adata = adata_all[mask].copy()
        if adata.n_obs == 0:
            raise ValueError(f"No cells were assigned to broad family {config['subset_family']}")
    print(f'Subset size: {adata.n_obs:,} tubules')

    expected_rows = []
    for role in ['early', 'late']:
        requested = config['position_markers'][role]
        genes = [gene_lookup[g.upper()] for g in requested if g.upper() in gene_lookup]
        missing = [g for g in requested if g.upper() not in gene_lookup]
        print(f'{prefix} {role} markers present ({len(genes)}/{len(requested)}): {genes}')
        if missing:
            print(f'  missing {role}: {missing}')
        if not genes:
            raise ValueError(f'No {role} position markers found for {prefix}')
        score_col = f'{prefix}_{role}_trajectory_score'
        sc.tl.score_genes(adata, genes, score_name=score_col, use_raw=False)
        for gene in genes:
            expected_rows.append({
                'gene': gene,
                'role': role,
                'expected_sign': -1 if role == 'early' else 1,
                'expected_direction': 'early-high' if role == 'early' else 'late-high',
            })

    marker_axis = adata.obs[f'{prefix}_late_trajectory_score'].astype(float) - adata.obs[f'{prefix}_early_trajectory_score'].astype(float)
    adata.obs[f'{prefix}_marker_axis'] = marker_axis
    marker_values = marker_axis.to_numpy(dtype=float)
    finite = np.isfinite(marker_values)
    normalized = np.full(marker_values.shape, np.nan, dtype=float)
    if finite.sum() > 0:
        p5, p95 = np.nanpercentile(marker_values[finite], [5, 95])
        normalized[finite] = np.clip((marker_values[finite] - p5) / max(p95 - p5, 1e-8), 0, 1)
    adata.obs[f'{prefix}_marker_axis_pseudotime'] = normalized

    saved_ps_col = f'{prefix}_pseudospace'
    if saved_ps_col not in adata.obs.columns:
        existing = next((col for col in ['pseudospace', f'{prefix}_position_percentile'] if col in adata.obs.columns), None)
        adata.obs[saved_ps_col] = pd.to_numeric(adata.obs[existing], errors='coerce') if existing is not None else adata.obs[f'{prefix}_marker_axis_pseudotime']

    marker_call_col = f'{prefix}_segment_marker_call'
    if marker_call_col not in adata.obs.columns:
        score_cols = []
        for segment, marker_genes in config['segment_markers'].items():
            genes = [gene_lookup[g.upper()] for g in marker_genes if g.upper() in gene_lookup]
            if not genes:
                continue
            score_col = f'{segment}_trajectory_compare_score'
            sc.tl.score_genes(adata, genes, score_name=score_col, use_raw=False)
            score_cols.append(score_col)
        if score_cols:
            scores = adata.obs[score_cols].astype(float)
            calls = scores.idxmax(axis=1).str.replace('_trajectory_compare_score', '', regex=False)
            adata.obs[marker_call_col] = pd.Categorical(calls, categories=config['segment_order'], ordered=True)

    method_cols = []
    method_notes = {}

    col = f'{prefix}_marker_axis_score'
    adata.obs[col] = adata.obs[f'{prefix}_marker_axis_pseudotime'].to_numpy(dtype=float)
    method_cols.append(col)
    method_notes[col] = 'marker-axis reference from early and late marker scores'

    values = adata.obs[saved_ps_col].to_numpy(dtype=float)
    reference = adata.obs[f'{prefix}_marker_axis'].to_numpy(dtype=float)
    valid = np.isfinite(values) & np.isfinite(reference)
    oriented = values.copy()
    if valid.sum() >= 10:
        corr = spearmanr(oriented[valid], reference[valid]).correlation
        if np.isfinite(corr) and corr < 0:
            oriented = -oriented
    finite = np.isfinite(oriented)
    normalized = np.full(oriented.shape, np.nan, dtype=float)
    if finite.sum() > 0:
        p5, p95 = np.nanpercentile(oriented[finite], [5, 95])
        normalized[finite] = np.clip((oriented[finite] - p5) / max(p95 - p5, 1e-8), 0, 1)
    col = f'{prefix}_saved_marker_pseudospace'
    adata.obs[col] = normalized
    method_cols.append(col)
    method_notes[col] = 'saved marker-derived pseudospace oriented to marker axis'

    preprocessed_loaded[continuum] = {
        'adata': adata,
        'config': config,
        'expected_markers': pd.DataFrame(expected_rows).drop_duplicates('gene').reset_index(drop=True),
        'method_cols': method_cols,
        'method_notes': method_notes,
    }
display(pd.DataFrame([{'continuum': k, 'n_tubules': v['adata'].n_obs, 'n_features': v['adata'].n_vars} for k, v in preprocessed_loaded.items()]))

### Clone Prepared Continua Helper

Defines a small helper that gives each trajectory method an independent copy of the prepared continuum objects.


In [ ]:
def clone_preprocessed_continua():
    cloned = {}
    for continuum, obj in preprocessed_loaded.items():
        cloned[continuum] = {
            'adata': obj['adata'].copy(),
            'config': obj['config'],
            'expected_markers': obj['expected_markers'].copy(),
            'method_cols': obj['method_cols'][:],
            'method_notes': obj['method_notes'].copy(),
        }
    return cloned
print('Prepared-continuum clone helper is defined.')

## PAGA Analysis


In [ ]:
# Use PAGA to check whether marker-defined discrete clusters form the expected topology.
paga_topology_rows = []

for continuum, obj in preprocessed_loaded.items():
    print('\n' + '=' * 80)
    print(f'Running PAGA topology check for {continuum}')
    adata = obj['adata']
    config = obj['config']
    prefix = config['prefix']
    cluster_col = f'{prefix}_segment_marker_call'

    if cluster_col not in adata.obs.columns:
        print(f'Skipped PAGA for {continuum}: missing {cluster_col}')
        continue

    labels = adata.obs[cluster_col].astype(str).replace({'nan': 'unassigned', 'None': 'unassigned'})
    present = pd.Index(labels.unique())
    group_order = [group for group in config['segment_order'] if group in present]
    group_order += sorted([group for group in present if group not in group_order])
    if len(group_order) < 2:
        print(f'Skipped PAGA for {continuum}: fewer than two discrete groups')
        continue
    paga_group_col = f'{prefix}_paga_group'
    adata.obs[paga_group_col] = pd.Categorical(labels, categories=group_order, ordered=True)

    preferred_reps = [
        f'X_pca_harmony_{prefix}',
        f'X_umap_harmony_{prefix}',
        'X_pca_harmony',
        'X_harmony',
        'X_pca',
    ]
    use_rep = next((rep for rep in preferred_reps if rep in adata.obsm), None)
    if use_rep is None:
        sc.pp.pca(adata, n_comps=min(N_PCS_FALLBACK, adata.n_vars - 1), random_state=RANDOM_STATE)
        use_rep = 'X_pca'
    print(f'{prefix}: PAGA neighbor graph uses representation {use_rep}')

    neighbors_key = f'{prefix}_paga_neighbors'
    paga_key = f'{prefix}_paga'
    sc.pp.neighbors(adata, n_neighbors=N_NEIGHBORS, use_rep=use_rep, key_added=neighbors_key, random_state=RANDOM_STATE)
    sc.tl.paga(adata, groups=paga_group_col, neighbors_key=neighbors_key, copy=False)
    adata.uns[paga_key] = adata.uns['paga'].copy()

    connectivities = adata.uns[paga_key]['connectivities']
    dense = connectivities.toarray() if sparse.issparse(connectivities) else np.asarray(connectivities)
    conn_df = pd.DataFrame(dense, index=group_order, columns=group_order)
    display(conn_df.round(3))

    expected_edges = list(zip(config['segment_order'][:-1], config['segment_order'][1:]))
    for left, right in expected_edges:
        if left not in conn_df.index or right not in conn_df.columns:
            continue
        connection = float(conn_df.loc[left, right])
        paga_topology_rows.append({
            'continuum': continuum,
            'edge_type': 'expected_adjacent',
            'left_group': left,
            'right_group': right,
            'paga_connectivity': connection,
        })

    expected_sets = {frozenset(edge) for edge in expected_edges}
    nonadjacent_rows = []
    for i, left in enumerate(group_order):
        for right in group_order[i + 1:]:
            if frozenset([left, right]) in expected_sets:
                continue
            nonadjacent_rows.append((left, right, float(conn_df.loc[left, right])))
    if nonadjacent_rows:
        left, right, connection = max(nonadjacent_rows, key=lambda row: row[2])
        paga_topology_rows.append({
            'continuum': continuum,
            'edge_type': 'strongest_nonadjacent',
            'left_group': left,
            'right_group': right,
            'paga_connectivity': connection,
        })

    fig, ax = plt.subplots(figsize=(5, 4))
    sc.pl.paga(
        adata,
        threshold=0.03,
        color=paga_group_col,
        title=f'{continuum} PAGA topology',
        show=False,
        ax=ax,
    )
    plt.show()

paga_topology_summary = pd.DataFrame(paga_topology_rows)
if not paga_topology_summary.empty:
    display(paga_topology_summary.sort_values(['continuum', 'edge_type', 'paga_connectivity'], ascending=[True, True, False]))
else:
    print('No PAGA topology summaries were generated')


## Method 1: Diffusion Pseudotime


In [ ]:
# dpt_base_loaded = clone_preprocessed_continua()
# dpt_loaded = {'TOTAL': dpt_base_loaded['TOTAL']}
# dpt_adata_all = adata_all.copy()

# # Run Scanpy diffusion map and DPT once on the total pseudospace.
# for continuum, obj in dpt_loaded.items():
#     print('\n' + '=' * 80)
#     print(f'Running Scanpy diffusion pseudotime for {continuum}')
#     adata = obj['adata']
#     config = obj['config']
#     prefix = config['prefix']
#     marker_axis = adata.obs[f'{prefix}_marker_axis'].to_numpy(dtype=float)

#     # Prefer harmonized embeddings; compute PCA only if no suitable representation exists.
#     preferred_reps = [
#         f'X_pca_harmony_{prefix}',
#         f'X_umap_harmony_{prefix}',
#         'X_pca_harmony',
#         'X_harmony',
#         'X_pca',
#     ]
#     use_rep = next((rep for rep in preferred_reps if rep in adata.obsm), None)
#     if use_rep is None:
#         sc.pp.pca(adata, n_comps=min(N_PCS_FALLBACK, adata.n_vars - 1), random_state=RANDOM_STATE)
#         use_rep = 'X_pca'
#     print(f'{prefix}: neighbor graph uses representation {use_rep}')

#     # Build the neighbor graph used by both diffusion map and DPT.
#     neighbors_key = 'trajectory_compare_neighbors'
#     sc.pp.neighbors(adata, n_neighbors=N_NEIGHBORS, use_rep=use_rep, key_added=neighbors_key, random_state=RANDOM_STATE)
#     sc.tl.diffmap(adata, neighbors_key=neighbors_key)

#     # Find which diffusion component best follows the marker-defined direction.
#     component_rows = []
#     n_components = min(N_DIFFMAP_COMPONENTS_TO_TEST, adata.obsm['X_diffmap'].shape[1])
#     for i in range(n_components):
#         values = adata.obsm['X_diffmap'][:, i].astype(float)
#         valid = np.isfinite(values) & np.isfinite(marker_axis)
#         corr = spearmanr(values[valid], marker_axis[valid]).correlation if valid.sum() >= 10 else np.nan
#         component_rows.append({'component': i, 'spearman_with_marker_axis': corr})
#     component_df = pd.DataFrame(component_rows)
#     display(component_df)

#     # Keep the most marker-aligned diffusion component as a comparable pseudotime-like coordinate.
#     best_row = component_df.iloc[component_df['spearman_with_marker_axis'].abs().fillna(-1).argmax()]
#     best_component = int(best_row['component'])
#     values = adata.obsm['X_diffmap'][:, best_component].astype(float)
#     valid = np.isfinite(values) & np.isfinite(marker_axis)
#     oriented = values.copy()
#     if valid.sum() >= 10:
#         corr = spearmanr(oriented[valid], marker_axis[valid]).correlation
#         if np.isfinite(corr) and corr < 0:
#             oriented = -oriented
#     finite = np.isfinite(oriented)
#     normalized = np.full(oriented.shape, np.nan, dtype=float)
#     if finite.sum() > 0:
#         p5, p95 = np.nanpercentile(oriented[finite], [5, 95])
#         normalized[finite] = np.clip((oriented[finite] - p5) / max(p95 - p5, 1e-8), 0, 1)
#     col = f'{prefix}_diffmap_component_{best_component}'
#     adata.obs[col] = normalized
#     obj['method_cols'].append(col)
#     obj['method_notes'][col] = f'Scanpy diffusion map component {best_component}, oriented to marker axis'

#     # Root DPT at the cell with the lowest marker-axis value, representing the earliest state.
#     root_candidates = np.where(np.isfinite(marker_axis))[0]
#     adata.uns['iroot'] = int(root_candidates[np.nanargmin(marker_axis[root_candidates])])
#     sc.tl.dpt(adata, neighbors_key=neighbors_key)

#     # Orient and normalize DPT so all methods share a 0-to-1 early-to-late scale.
#     values = adata.obs['dpt_pseudotime'].to_numpy(dtype=float)
#     valid = np.isfinite(values) & np.isfinite(marker_axis)
#     oriented = values.copy()
#     if valid.sum() >= 10:
#         corr = spearmanr(oriented[valid], marker_axis[valid]).correlation
#         if np.isfinite(corr) and corr < 0:
#             oriented = -oriented
#     finite = np.isfinite(oriented)
#     normalized = np.full(oriented.shape, np.nan, dtype=float)
#     if finite.sum() > 0:
#         p5, p95 = np.nanpercentile(oriented[finite], [5, 95])
#         normalized[finite] = np.clip((oriented[finite] - p5) / max(p95 - p5, 1e-8), 0, 1)
#     col = f'{prefix}_scanpy_dpt'
#     adata.obs[col] = normalized
#     obj['method_cols'].append(col)
#     obj['method_notes'][col] = f'Scanpy DPT on {use_rep}, rooted at lowest marker-axis cell'

# # Plot total diffusion pseudotime on the available UMAP embedding.
# total_obj = dpt_loaded['TOTAL']
# total_adata = total_obj['adata']
# total_dpt_col = 'total_scanpy_dpt'

# basis = None
# for key in ['X_umap_harmony_total', 'X_umap_harmony', 'X_umap']:
#     if key in total_adata.obsm:
#         basis = key.replace('X_', '', 1)
#         break
# if basis is None:
#     for key in total_adata.obsm.keys():
#         if str(key).startswith('X_umap'):
#             basis = str(key).replace('X_', '', 1)
#             break
# if basis is None:
#     sc.tl.umap(total_adata, neighbors_key='trajectory_compare_neighbors', random_state=RANDOM_STATE)
#     basis = 'umap'

# sc.pl.embedding(
#     total_adata,
#     basis=basis,
#     color=total_dpt_col,
#     cmap='viridis',
#     frameon=False,
#     na_color='lightgray',
#     size=8 if total_adata.n_obs < 5000 else 3,
#     title='TOTAL Scanpy DPT',
# )



In [ ]:
# Summary of changes
# Removed UMAP from preferred_reps — UMAP is now only used as a plotting basis, never for graph construction.
# choose_root helper — picks the medoid of the bottom-1% marker-axis cells in the chosen representation, avoiding the single-outlier failure mode of argmin.
# choose_diffusion_component helper — scores components by |spearman| × eigenvalue with an eigenvalue floor of 0.5, so low-eigenvalue noise modes can't win on a fluke correlation.
# orient_and_normalize helper — factored out the duplicated sign-flip + p5/p95 clip; raised the Spearman validity threshold from 10 to 100 samples.
# Diffusion component stored un-clipped — it is a signed coordinate, not a [0, 1] pseudotime; only the sign is aligned to the marker axis.
# PCA fallback — n_comps is now bounded by min(N_PCS_FALLBACK, n_obs - 1, n_vars - 1).
# Adaptive n_neighbors — scales as max(N_NEIGHBORS, √n_obs) so global runs aren't fragmented and within-segment runs aren't over-smoothed.
# Root identity logged — both the cell index and the marker-axis value are stored in method_notes so the choice is auditable later.
# Component eigenvalue logged in method_notes for the same reason.

# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

MIN_VALID_FOR_SPEARMAN = 100
EIGENVALUE_FLOOR = 0.5  # diffusion components with eigenvalues below this are treated as noise


def orient_and_normalize(values, marker_axis, min_valid=MIN_VALID_FOR_SPEARMAN):
    """Flip sign so values increase with marker axis, then clip to p5–p95 and rescale to [0, 1]."""
    values = np.asarray(values, dtype=float)
    marker_axis = np.asarray(marker_axis, dtype=float)
    valid = np.isfinite(values) & np.isfinite(marker_axis)
    oriented = values.copy()
    if valid.sum() >= min_valid:
        corr = spearmanr(oriented[valid], marker_axis[valid]).correlation
        if np.isfinite(corr) and corr < 0:
            oriented = -oriented
    finite = np.isfinite(oriented)
    out = np.full(oriented.shape, np.nan, dtype=float)
    if finite.any():
        p5, p95 = np.nanpercentile(oriented[finite], [5, 95])
        out[finite] = np.clip((oriented[finite] - p5) / max(p95 - p5, 1e-8), 0, 1)
    return out


def choose_representation(adata, prefix):
    """Prefer harmonized PCA representations; never use UMAP for graph construction."""
    preferred_reps = [
        f'X_pca_harmony_{prefix}',
        'X_pca_harmony',
        'X_harmony',
        'X_pca',
    ]
    for rep in preferred_reps:
        if rep in adata.obsm:
            return rep
    n_comps = min(N_PCS_FALLBACK, adata.n_obs - 1, adata.n_vars - 1)
    sc.pp.pca(adata, n_comps=n_comps, random_state=RANDOM_STATE)
    return 'X_pca'


def choose_root(marker_axis, adata, rep_key, bottom_quantile=0.01):
    """Root = medoid (in the chosen representation) of the bottom-quantile marker-axis cells.

    This is more robust than argmin, which is dominated by a single outlier.
    """
    finite_mask = np.isfinite(marker_axis)
    if not finite_mask.any():
        raise ValueError('marker_axis has no finite values; cannot choose a root.')
    threshold = np.nanpercentile(marker_axis[finite_mask], 100 * bottom_quantile)
    candidates = np.where(finite_mask & (marker_axis <= threshold))[0]
    if candidates.size == 1:
        return int(candidates[0])
    coords = adata.obsm[rep_key][candidates]
    centroid = coords.mean(axis=0, keepdims=True)
    dists = np.linalg.norm(coords - centroid, axis=1)
    return int(candidates[int(np.argmin(dists))])


def choose_diffusion_component(adata, marker_axis, n_components_to_test, eigenvalue_floor=EIGENVALUE_FLOOR):
    """Pick the diffusion component that best follows the marker axis, weighted by eigenvalue."""
    diffmap = adata.obsm['X_diffmap']
    eigvals = np.asarray(adata.uns['diffmap_evals'], dtype=float)
    n_components = min(n_components_to_test, diffmap.shape[1], eigvals.size)

    rows = []
    for i in range(n_components):
        values = diffmap[:, i].astype(float)
        valid = np.isfinite(values) & np.isfinite(marker_axis)
        corr = (
            spearmanr(values[valid], marker_axis[valid]).correlation
            if valid.sum() >= MIN_VALID_FOR_SPEARMAN
            else np.nan
        )
        rows.append({
            'component': i,
            'eigenvalue': float(eigvals[i]),
            'spearman_with_marker_axis': corr,
            'score': (abs(corr) * eigvals[i]) if (np.isfinite(corr) and eigvals[i] >= eigenvalue_floor) else np.nan,
        })
    component_df = pd.DataFrame(rows)

    if component_df['score'].notna().any():
        best_component = int(component_df.loc[component_df['score'].idxmax(), 'component'])
    else:
        # Fallback: ignore eigenvalue floor if nothing passes it.
        fallback = component_df['spearman_with_marker_axis'].abs().fillna(-1)
        best_component = int(component_df.loc[fallback.idxmax(), 'component'])

    return best_component, component_df


# ---------------------------------------------------------------------------
# Run diffusion map + DPT per continuum
# ---------------------------------------------------------------------------

dpt_base_loaded = clone_preprocessed_continua()
dpt_loaded = {'TOTAL': dpt_base_loaded['TOTAL']}

for continuum, obj in dpt_loaded.items():
    print('\n' + '=' * 80)
    print(f'Running Scanpy diffusion pseudotime for {continuum}')
    adata = obj['adata']
    config = obj['config']
    prefix = config['prefix']
    marker_axis = adata.obs[f'{prefix}_marker_axis'].to_numpy(dtype=float)

    # Representation (harmonized PCA preferred; never UMAP).
    use_rep = choose_representation(adata, prefix)
    print(f'{prefix}: neighbor graph uses representation {use_rep}')

    # Neighbor count scales with sample size; floor at 15 for global runs.
    n_neighbors = max(N_NEIGHBORS, int(np.sqrt(adata.n_obs)))
    print(f'{prefix}: n_neighbors = {n_neighbors} (n_obs = {adata.n_obs})')

    neighbors_key = 'trajectory_compare_neighbors'
    sc.pp.neighbors(
        adata,
        n_neighbors=n_neighbors,
        use_rep=use_rep,
        key_added=neighbors_key,
        random_state=RANDOM_STATE,
    )
    sc.tl.diffmap(adata, neighbors_key=neighbors_key)

    # Pick the most marker-aligned diffusion component (weighted by eigenvalue).
    best_component, component_df = choose_diffusion_component(
        adata, marker_axis, n_components_to_test=N_DIFFMAP_COMPONENTS_TO_TEST
    )
    display(component_df)
    best_eigval = float(component_df.loc[component_df['component'] == best_component, 'eigenvalue'].iloc[0])
    print(f'{prefix}: best diffusion component = {best_component} (eigenvalue = {best_eigval:.3f})')

    # Store the chosen component, sign-flipped to marker axis (no clipping — it is a signed coordinate).
    component_values = adata.obsm['X_diffmap'][:, best_component].astype(float)
    valid = np.isfinite(component_values) & np.isfinite(marker_axis)
    if valid.sum() >= MIN_VALID_FOR_SPEARMAN:
        corr = spearmanr(component_values[valid], marker_axis[valid]).correlation
        if np.isfinite(corr) and corr < 0:
            component_values = -component_values
    col = f'{prefix}_diffmap_component_{best_component}'
    adata.obs[col] = component_values
    obj['method_cols'].append(col)
    obj['method_notes'][col] = (
        f'Scanpy diffusion map component {best_component} '
        f'(eigenvalue {best_eigval:.3f}), oriented to marker axis; not rescaled.'
    )

    # Robust root: medoid of bottom-1% marker-axis cells in the chosen representation.
    iroot = choose_root(marker_axis, adata, use_rep, bottom_quantile=0.01)
    adata.uns['iroot'] = iroot
    root_marker_value = float(marker_axis[iroot])
    root_identity = adata.obs.index[iroot]
    print(f'{prefix}: root = {root_identity} (idx {iroot}, marker_axis = {root_marker_value:.3f})')

    # DPT.
    sc.tl.dpt(adata, neighbors_key=neighbors_key)
    dpt_values = adata.obs['dpt_pseudotime'].to_numpy(dtype=float)
    normalized = orient_and_normalize(dpt_values, marker_axis)
    col = f'{prefix}_scanpy_dpt'
    adata.obs[col] = normalized
    obj['method_cols'].append(col)
    obj['method_notes'][col] = (
        f'Scanpy DPT on {use_rep} (n_neighbors={n_neighbors}); '
        f'root = {root_identity} (idx {iroot}, marker_axis = {root_marker_value:.3f}); '
        f'oriented to marker axis and clipped to p5–p95.'
    )


# ---------------------------------------------------------------------------
# Plot TOTAL DPT on a UMAP embedding (UMAP is only used for visualization here)
# ---------------------------------------------------------------------------

total_obj = dpt_loaded['TOTAL']
total_adata = total_obj['adata']
total_dpt_col = 'total_scanpy_dpt'

basis = None
for key in ['X_umap_harmony_total', 'X_umap_harmony', 'X_umap']:
    if key in total_adata.obsm:
        basis = key.replace('X_', '', 1)
        break
if basis is None:
    for key in total_adata.obsm.keys():
        if str(key).startswith('X_umap'):
            basis = str(key).replace('X_', '', 1)
            break
if basis is None:
    sc.tl.umap(total_adata, neighbors_key='trajectory_compare_neighbors', random_state=RANDOM_STATE)
    basis = 'umap'

sc.pl.embedding(
    total_adata,
    basis=basis,
    color=total_dpt_col,
    cmap='viridis',
    frameon=False,
    na_color='lightgray',
    size=8 if total_adata.n_obs < 5000 else 3,
    title='TOTAL Scanpy DPT',
)

In [ ]:
from scipy.stats import spearmanr

PT_MARKER_GROUPS = {
    'PT-S1': [
        {'label': 'SLC5A2 / Slc5a2', 'candidates': ['SLC5A2', 'Slc5a2']},
        {'label': 'SLC5A12 / Slc5a12', 'candidates': ['SLC5A12', 'Slc5a12']},
    ],
    'PT-S2': [
        {'label': 'SLC22A8 / Slc22a8', 'candidates': ['SLC22A8', 'Slc22a8']},
        {'label': 'SLC22A6 / Slc22a6', 'candidates': ['SLC22A6', 'Slc22a6']},
        {'label': 'SLC13A3 / Slc13a3', 'candidates': ['SLC13A3', 'Slc13a3']},
        {'label': 'SLC22A2 / Slc22a2', 'candidates': ['SLC22A2', 'Slc22a2']},
        {'label': 'SLC34A1 / Slc34a1', 'candidates': ['SLC34A1', 'Slc34a1']},
    ],
    'PT-S3': [
        {'label': 'SLC7A13 / Slc7a13', 'candidates': ['SLC7A13', 'Slc7a13']},
        {'label': 'SLC5A8 / Slc5a8', 'candidates': ['SLC5A8', 'Slc5a8']},
        {'label': 'SLC38A3 / Slc38a3', 'candidates': ['SLC38A3', 'Slc38a3']},
        {'label': 'HAO2 / Hao2', 'candidates': ['HAO2', 'Hao2']},
        {'label': 'PCK1 / Pck1', 'candidates': ['PCK1', 'Pck1']},
        {'label': 'ACSM3 / Acsm3', 'candidates': ['ACSM3', 'Acsm3']},
    ],
}
PT_GROUP_ORDER = ['PT-S1', 'PT-S2', 'PT-S3']
PT_GROUP_COLORS = {'PT-S1': '#4C9BD3', 'PT-S2': '#5B8E55', 'PT-S3': '#D6B48A'}
PT_COLORS = PT_GROUP_COLORS

TAL_MARKER_GROUPS = {
    'mTAL / early TAL-like': [
        {'label': 'SLC12A1 / Slc12a1', 'candidates': ['SLC12A1', 'Slc12a1']},
        {'label': 'UMOD / Umod', 'candidates': ['UMOD', 'Umod']},
        {'label': 'KCNJ1 / Kcnj1', 'candidates': ['KCNJ1', 'Kcnj1']},
        {'label': 'BSND / Bsnd', 'candidates': ['BSND', 'Bsnd']},
        {'label': 'CLCNKB / Clcnkb', 'candidates': ['CLCNKB', 'Clcnkb']},
        {'label': 'ATP1A1 / Atp1a1', 'candidates': ['ATP1A1', 'Atp1a1']},
        {'label': 'ATP1B1 / Atp1b1', 'candidates': ['ATP1B1', 'Atp1b1']},
    ],
    'cTAL / late TAL-like': [
        {'label': 'CLDN10 / Cldn10', 'candidates': ['CLDN10', 'Cldn10']},
        {'label': 'CLDN16 / Cldn16', 'candidates': ['CLDN16', 'Cldn16']},
        {'label': 'CLDN19 / Cldn19', 'candidates': ['CLDN19', 'Cldn19']},
        {'label': 'CASR / Casr', 'candidates': ['CASR', 'Casr']},
        {'label': 'PTH1R / Pth1r', 'candidates': ['PTH1R', 'Pth1r']},
        {'label': 'VDR / Vdr', 'candidates': ['VDR', 'Vdr']},
    ],
    'Macula densa / TAL-DCT boundary': [
        {'label': 'NOS1 / Nos1', 'candidates': ['NOS1', 'Nos1']},
        {'label': 'PTGS2 / Ptgs2', 'candidates': ['PTGS2', 'Ptgs2']},
    ],
}
TAL_GROUP_ORDER = ['mTAL / early TAL-like', 'cTAL / late TAL-like', 'Macula densa / TAL-DCT boundary']
TAL_GROUP_COLORS = {'mTAL / early TAL-like': '#4C78A8', 'cTAL / late TAL-like': '#F58518', 'Macula densa / TAL-DCT boundary': '#E45756'}
TAL_COLORS = TAL_GROUP_COLORS

DCT_MARKER_GROUPS = {
    'DCT1': [
        {'label': 'SLC12A3 / Slc12a3', 'candidates': ['SLC12A3', 'Slc12a3']},
        {'label': 'PVALB / Pvalb', 'candidates': ['PVALB', 'Pvalb']},
    ],
    'DCT2': [
        {'label': 'CALB1 / Calb1', 'candidates': ['CALB1', 'Calb1']},
        {'label': 'HSD11B2 / Hsd11b2', 'candidates': ['HSD11B2', 'Hsd11b2']},
        {'label': 'SCNN1G / Scnn1g', 'candidates': ['SCNN1G', 'Scnn1g']},
    ],
}
DCT_GROUP_ORDER = ['DCT1', 'DCT2']
DCT_GROUP_COLORS = {'DCT1': '#4C78A8', 'DCT2': '#F58518'}
DCT_COLORS = DCT_GROUP_COLORS

TOTAL_NEPHRON_MARKERS = [
    {'label': 'Slc5a2 (PT-S1)', 'candidates': ['SLC5A2', 'Slc5a2']},
    {'label': 'Slc22a6 (PT-S2)', 'candidates': ['SLC22A6', 'Slc22a6']},
    {'label': 'Lrp2 (PT / pan-PT)', 'candidates': ['LRP2', 'Lrp2']},
    {'label': 'Slc9a3 (PT)', 'candidates': ['SLC9A3', 'Slc9a3']},
    {'label': 'Slc34a1 (PT / phosphate transport)', 'candidates': ['SLC34A1', 'Slc34a1']},
    {'label': 'Aqp1 (DTL / PT-DTL-associated, depending subset)', 'candidates': ['AQP1', 'Aqp1']},
    {'label': 'Clcnka (ATL)', 'candidates': ['CLCNKA', 'Clcnka']},
    {'label': 'Slc12a1 (TAL)', 'candidates': ['SLC12A1', 'Slc12a1']},
    {'label': 'Umod (TAL)', 'candidates': ['UMOD', 'Umod']},
    {'label': 'Pvalb (DCT1)', 'candidates': ['PVALB', 'Pvalb']},
    {'label': 'Slc12a3 (DCT)', 'candidates': ['SLC12A3', 'Slc12a3']},
    {'label': 'Calb1 (DCT2/CNT)', 'candidates': ['CALB1', 'Calb1']},
    {'label': 'Aqp2 (CCD / principal cell)', 'candidates': ['AQP2', 'Aqp2']},
    {'label': 'Scnn1g (CNT/CD principal cell)', 'candidates': ['SCNN1G', 'Scnn1g']},
    {'label': 'Slc4a1 (OMCD / intercalated cell)', 'candidates': ['SLC4A1', 'Slc4a1']},
    {'label': 'Slc26a4 (OMCD / pendrin+ intercalated cell)', 'candidates': ['SLC26A4', 'Slc26a4']},
    {'label': 'Atp6v1g3 (CD intercalated cell)', 'candidates': ['ATP6V1G3', 'Atp6v1g3']},
]
TOTAL_MARKERS = TOTAL_NEPHRON_MARKERS
print('Marker panels defined for PT, TAL, DCT, and TOTAL heatmaps.')

def matrix_to_dense(matrix):
    if sparse.issparse(matrix):
        return matrix.toarray()
    return np.asarray(matrix)

def resolve_gene_aliases(adata_obj, marker_groups):
    var_names = pd.Index(adata_obj.var_names.astype(str))
    lookup = {name.upper(): name for name in var_names}
    alias_columns = [col for col in ['gene_symbol', 'gene_symbols', 'symbol', 'gene_name', 'feature_name', 'features'] if col in adata_obj.var.columns]
    for col in alias_columns:
        for idx_name, alias in zip(var_names, adata_obj.var[col].astype(str)):
            alias_upper = alias.upper()
            if alias_upper and alias_upper != 'NAN' and alias_upper not in lookup:
                lookup[alias_upper] = idx_name
    resolved = {}
    missing = []
    for group_name, markers in marker_groups.items():
        for marker in markers:
            match = None
            for candidate in marker['candidates']:
                match = lookup.get(candidate.upper())
                if match is not None:
                    break
            if match is None:
                missing.append(marker['label'])
            else:
                resolved[marker['label']] = match
    return resolved, missing

def resolve_marker_list(adata_obj, markers):
    resolved, missing = resolve_gene_aliases(adata_obj, {'all': markers})
    ordered = [marker['label'] for marker in markers if marker['label'] in resolved]
    return ordered, resolved, missing

print('Gene-resolution helper functions are defined.')


def marker_labels_for_group(marker_groups, group_order):
    labels = []
    for group in group_order:
        for marker in marker_groups.get(group, []):
            labels.append(marker['label'])
    return labels


def resolve_available_marker_groups(adata_obj, marker_groups, group_order):
    resolved_map, missing = resolve_gene_aliases(adata_obj, marker_groups)
    available_groups = {}
    for group in group_order:
        keep = [marker['label'] for marker in marker_groups.get(group, []) if marker['label'] in resolved_map]
        if keep:
            available_groups[group] = keep
    if not available_groups:
        preview_cols = [
            col for col in ['gene_symbol', 'gene_symbols', 'symbol', 'gene_name', 'feature_name', 'features']
            if col in adata_obj.var.columns
        ]
        raise ValueError(
            'None of the requested marker genes were found. '
            f'Checked adata.var_names and alias columns: {preview_cols if preview_cols else "none"}.'
        )
    gene_order = [gene for group in group_order if group in available_groups for gene in available_groups[group]]
    gene_idx = [adata_obj.var_names.get_loc(resolved_map[gene]) for gene in gene_order]
    return available_groups, gene_order, gene_idx, missing


def binned_interpolated_expression(adata_obj, gene_idx, pseudotime_col, n_bins=120, smooth_sigma=2.5, expression_layer='lognorm'):
    pos = adata_obj.obs[pseudotime_col].to_numpy(dtype=float)
    finite = np.isfinite(pos)
    if finite.sum() < 10:
        raise ValueError(f'Too few finite {pseudotime_col} values to build heatmap')
    adata_obj = adata_obj[finite].copy()
    pos = pos[finite]

    x_min = float(np.nanmin(pos))
    x_max = float(np.nanmax(pos))
    if np.isclose(x_min, x_max):
        x_min, x_max = x_min - 0.005, x_max + 0.005

    sort_idx = np.argsort(pos)
    pos = pos[sort_idx]
    X_obj = adata_obj.layers[expression_layer] if expression_layer in adata_obj.layers else adata_obj.X
    expr = matrix_to_dense(X_obj[:, gene_idx])[sort_idx].T

    bin_edges = np.linspace(x_min, x_max, n_bins + 1)
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    bin_ids = np.digitize(pos, bin_edges) - 1
    bin_ids = np.clip(bin_ids, 0, n_bins - 1)

    heatmap = np.full((len(gene_idx), n_bins), np.nan)
    occupied = np.zeros(n_bins, dtype=bool)
    for bin_i in range(n_bins):
        mask = bin_ids == bin_i
        count = int(mask.sum())
        if count > 0:
            occupied[bin_i] = True
            heatmap[:, bin_i] = expr[:, mask].mean(axis=1)

    occupied_idx = np.flatnonzero(occupied)
    if len(occupied_idx) == 0:
        raise ValueError(f'No occupied bins for {pseudotime_col}')
    first_bin = int(occupied_idx[0])
    last_bin = int(occupied_idx[-1])

    # Interpolate every internal empty bin without compressing the x coordinate.
    for row_i in range(heatmap.shape[0]):
        valid = np.isfinite(heatmap[row_i])
        valid[:first_bin] = False
        valid[last_bin + 1:] = False
        if valid.sum() == 0:
            continue
        if valid.sum() == 1:
            heatmap[row_i, first_bin:last_bin + 1] = heatmap[row_i, valid][0]
        else:
            heatmap[row_i, first_bin:last_bin + 1] = np.interp(
                bin_centers[first_bin:last_bin + 1],
                bin_centers[valid],
                heatmap[row_i, valid],
            )

    heatmap = heatmap[:, first_bin:last_bin + 1]
    bin_centers = bin_centers[first_bin:last_bin + 1]
    bin_edges = bin_edges[first_bin:last_bin + 2]

    if smooth_sigma and smooth_sigma > 0:
        heatmap = gaussian_filter1d(heatmap, sigma=smooth_sigma, axis=1, mode='nearest')

    gene_mean = np.nanmean(heatmap, axis=1, keepdims=True)
    gene_std = np.nanstd(heatmap, axis=1, keepdims=True)
    gene_std[~np.isfinite(gene_std) | (gene_std == 0)] = 1
    heatmap_z = (heatmap - gene_mean) / gene_std
    return heatmap, heatmap_z, bin_centers, bin_edges, occupied[first_bin:last_bin + 1]


def plot_marker_heatmap(
    adata_obj,
    marker_groups,
    group_order,
    group_colors,
    pseudotime_col,
    cluster_col,
    cluster_value=None,
    title=None,
    output_name='marker_heatmap.png',
    mask=None,
    target_label=None,
    strip_col=None,
    strip_order=None,
    strip_colors=None,
    n_bins=120,
    smooth_sigma=2.5,
    trim_fraction=0.05,
    z_clip=2,
    expression_layer='lognorm',
    show_module_legend=True,
    show_group_labels=True,
    figsize=(16, 10),
):
    if pseudotime_col not in adata_obj.obs.columns:
        raise KeyError(f'Missing pseudotime column: {pseudotime_col}')
    if cluster_col is not None and cluster_col not in adata_obj.obs.columns:
        raise KeyError(f'Missing cluster label column from notebook 2: {cluster_col}')

    plot_mask = np.ones(adata_obj.n_obs, dtype=bool)
    if cluster_col is not None and cluster_value is not None:
        plot_mask &= adata_obj.obs[cluster_col].astype(str).eq(str(cluster_value)).to_numpy()
    if mask is not None:
        plot_mask &= np.asarray(mask, dtype=bool)
    plot_mask &= adata_obj.obs[pseudotime_col].notna().to_numpy()
    if plot_mask.sum() == 0:
        raise ValueError(f'No tubules available for {title or output_name}')

    plot_adata = adata_obj[plot_mask].copy()
    n_cells_before_trim = int(plot_adata.n_obs)
    if not 0 <= trim_fraction < 0.5:
        raise ValueError('trim_fraction must be in [0, 0.5)')
    n_trim_per_end = int(np.floor(n_cells_before_trim * trim_fraction))
    if n_trim_per_end > 0:
        pseudotime_order = np.argsort(
            plot_adata.obs[pseudotime_col].to_numpy(dtype=float),
            kind='stable',
        )
        keep_idx = pseudotime_order[n_trim_per_end:n_cells_before_trim - n_trim_per_end]
        plot_adata = plot_adata[keep_idx].copy()
    if plot_adata.n_obs < 10:
        raise ValueError(f'Too few tubules remain after pseudospace trimming for {title or output_name}')

    available_groups, gene_order, gene_idx, missing = resolve_available_marker_groups(plot_adata, marker_groups, group_order)
    heatmap, heatmap_z, bin_centers, bin_edges, occupied_bins = binned_interpolated_expression(
        plot_adata,
        gene_idx,
        pseudotime_col,
        n_bins=n_bins,
        smooth_sigma=smooth_sigma,
        expression_layer=expression_layer,
    )
    heatmap_z = np.clip(heatmap_z, -z_clip, z_clip)

    module_names = [group for group in group_order if group in available_groups]
    gene_pos = {gene: i for i, gene in enumerate(gene_order)}
    module_scores = []
    for module_name in module_names:
        idx = [gene_pos[gene] for gene in available_groups[module_name]]
        module_scores.append(np.nanmean(heatmap_z[idx], axis=0))
    module_scores = np.vstack(module_scores)
    dominant_module = np.nanargmax(module_scores, axis=0)

    strip_values = dominant_module
    strip_cmap = ListedColormap([group_colors[name] for name in module_names])
    if strip_col is not None:
        if strip_col not in plot_adata.obs.columns:
            raise KeyError(f'Missing cluster strip column from notebook 2: {strip_col}')
        labels = plot_adata.obs[strip_col].astype(str).to_numpy()
        pos = plot_adata.obs[pseudotime_col].to_numpy(dtype=float)
        if strip_order is None:
            if hasattr(plot_adata.obs[strip_col], 'cat'):
                strip_order = [str(x) for x in plot_adata.obs[strip_col].cat.categories]
            else:
                strip_order = sorted(pd.unique(labels).astype(str))
        strip_order = [str(x) for x in strip_order if str(x) not in {'nan', 'None'}]
        strip_lookup = {label: i for i, label in enumerate(strip_order)}
        strip_values = np.full(len(bin_centers), np.nan, dtype=float)
        for bin_i in range(len(bin_centers)):
            if bin_i == len(bin_centers) - 1:
                in_bin = (pos >= bin_edges[bin_i]) & (pos <= bin_edges[bin_i + 1])
            else:
                in_bin = (pos >= bin_edges[bin_i]) & (pos < bin_edges[bin_i + 1])
            bin_labels = pd.Series(labels[in_bin]).replace({'nan': np.nan, 'None': np.nan}).dropna()
            if len(bin_labels):
                label = str(bin_labels.value_counts().idxmax())
                if label in strip_lookup:
                    strip_values[bin_i] = strip_lookup[label]
        if strip_colors is None:
            palette = plt.get_cmap('tab20').colors
            strip_colors = {label: palette[i % len(palette)] for i, label in enumerate(strip_order)}
        strip_cmap = ListedColormap([strip_colors.get(label, '#4D4D4D') for label in strip_order])

    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(3, 1, height_ratios=[1.2, 0.25, 8], hspace=0.08)
    ax_top = fig.add_subplot(gs[0])
    ax_strip = fig.add_subplot(gs[1], sharex=ax_top)
    ax_heat = fig.add_subplot(gs[2], sharex=ax_top)

    for module_idx, module_name in enumerate(module_names):
        ax_top.plot(
            bin_centers,
            module_scores[module_idx],
            color=group_colors[module_name],
            linewidth=2.5,
            label=module_name,
        )
    ax_top.set_xlim(float(bin_edges[0]), float(bin_edges[-1]))
    ax_top.set_ylabel('Module score')
    ax_top.set_xticks([])
    ax_top.spines['top'].set_visible(False)
    ax_top.spines['right'].set_visible(False)
    if show_module_legend:
        ax_top.legend(
            loc='upper right',
            frameon=False,
            ncol=min(3, max(1, len(module_names))),
            fontsize=9,
        )

    ax_strip.imshow(
        strip_values[np.newaxis, :],
        aspect='auto',
        cmap=strip_cmap,
        interpolation='nearest',
        extent=[float(bin_edges[0]), float(bin_edges[-1]), 0, 1],
    )
    ax_strip.set_yticks([])
    ax_strip.set_xticks([])
    for spine in ax_strip.spines.values():
        spine.set_visible(False)

    im = ax_heat.imshow(
        heatmap_z,
        aspect='auto',
        cmap='bwr',
        vmin=-z_clip,
        vmax=z_clip,
        interpolation='nearest',
        extent=[float(bin_edges[0]), float(bin_edges[-1]), len(gene_order) - 0.5, -0.5],
    )
    ax_heat.set_xlim(float(bin_edges[0]), float(bin_edges[-1]))
    ax_heat.set_xlabel(pseudotime_col)
    xticks = np.linspace(float(bin_edges[0]), float(bin_edges[-1]), 6)
    ax_heat.set_xticks(xticks)
    ax_heat.set_xticklabels([f'{x:.2f}' for x in xticks])
    ax_heat.set_yticks(np.arange(len(gene_order)))
    ax_heat.set_yticklabels(gene_order, fontsize=10)
    ax_heat.yaxis.tick_right()
    ax_heat.tick_params(axis='y', length=0)

    if target_label:
        ax_heat.text(
            1.01,
            1.01,
            target_label,
            transform=ax_heat.transAxes,
            ha='left',
            va='bottom',
            fontsize=10,
            bbox={'boxstyle': 'round,pad=0.3', 'facecolor': 'white', 'edgecolor': 'black', 'linewidth': 0.8},
        )

    offset = 0
    for module_name in module_names:
        genes = available_groups[module_name]
        center = offset + (len(genes) - 1) / 2
        if show_group_labels:
            ax_heat.text(
                -0.02,
                center,
                module_name,
                transform=ax_heat.get_yaxis_transform(),
                ha='right',
                va='center',
                fontsize=11,
                fontweight='bold',
                color=group_colors[module_name],
            )
        offset += len(genes)
        if offset < len(gene_order):
            ax_heat.axhline(offset - 0.5, color='black', linewidth=0.8)

    fig.subplots_adjust(left=0.10, right=0.76, top=0.92, bottom=0.08, hspace=0.08)
    cbar = fig.colorbar(im, ax=ax_heat, fraction=0.03, pad=0.12)
    cbar.set_label('Per-gene z-score')
    fig.suptitle(title or output_name, fontsize=16, fontweight='bold', y=0.98)
    fig.savefig(OUTPUT_DIR / output_name, dpi=200, bbox_inches='tight')
    plt.show()

    return {
        'n_cells': int(plot_adata.n_obs),
        'n_cells_before_trim': n_cells_before_trim,
        'n_cells_trimmed_per_end': n_trim_per_end,
        'trim_fraction_per_end': float(trim_fraction),
        'cluster_col': cluster_col,
        'cluster_value': cluster_value,
        'pseudotime_col': pseudotime_col,
        'dpt_min': float(bin_edges[0]),
        'dpt_max': float(bin_edges[-1]),
        'genes_used': gene_order,
        'missing_markers': missing,
        'module_names': module_names,
        'output_path': str((OUTPUT_DIR / output_name).relative_to(PROJECT_DIR)),
    }

print('Shared marker heatmap function is defined.')


# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------

MIN_VALID_FOR_SPEARMAN = 100
EIGENVALUE_FLOOR = 0.5
N_DIFFMAP_COMPONENTS_TO_TEST = 10


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------

def robust_scale_rows(values):
    values = np.asarray(values, dtype=float)
    mean = np.nanmean(values, axis=1, keepdims=True)
    std = np.nanstd(values, axis=1, keepdims=True)
    std[~np.isfinite(std) | (std == 0)] = 1.0
    return (values - mean) / std


def orient_and_normalize(values, marker_axis, min_valid=MIN_VALID_FOR_SPEARMAN):
    """Flip sign so values increase with marker axis, then clip to p5–p95 and rescale to [0, 1]."""
    values = np.asarray(values, dtype=float)
    marker_axis = np.asarray(marker_axis, dtype=float)
    valid = np.isfinite(values) & np.isfinite(marker_axis)
    oriented = values.copy()
    orientation_corr = np.nan
    if valid.sum() >= min_valid:
        orientation_corr = spearmanr(oriented[valid], marker_axis[valid]).correlation
        if np.isfinite(orientation_corr) and orientation_corr < 0:
            oriented = -oriented
    finite = np.isfinite(oriented)
    out = np.full(oriented.shape, np.nan, dtype=float)
    if finite.any():
        p5, p95 = np.nanpercentile(oriented[finite], [5, 95])
        out[finite] = np.clip((oriented[finite] - p5) / max(p95 - p5, 1e-8), 0, 1)
    return out, float(orientation_corr) if np.isfinite(orientation_corr) else np.nan


def choose_subset_representation(subset):
    """Prefer harmonized PCA; never use UMAP for graph construction."""
    preferred_reps = ['X_pca_harmony', 'X_harmony', 'X_pca']
    for rep in preferred_reps:
        if rep in subset.obsm:
            return rep
    raise KeyError('No suitable representation (X_pca_harmony, X_harmony, X_pca) found in subset.obsm')


def choose_subset_root(marker_axis, subset, rep_key, bottom_quantile=0.05):
    """Root = medoid (in the chosen representation) of the bottom-quantile marker-axis tubules."""
    finite_mask = np.isfinite(marker_axis)
    if finite_mask.sum() < 10:
        raise ValueError('Too few finite marker-axis values to pick a root')
    threshold = np.nanquantile(marker_axis[finite_mask], bottom_quantile)
    candidates = np.flatnonzero(finite_mask & (marker_axis <= threshold))
    if candidates.size == 1:
        return int(candidates[0])
    coords = np.asarray(subset.obsm[rep_key], dtype=float)[candidates]
    centroid = np.nanmedian(coords, axis=0, keepdims=True)
    dists = np.linalg.norm(coords - centroid, axis=1)
    return int(candidates[int(np.nanargmin(dists))])


def choose_subset_diffusion_component(subset, marker_axis, n_components_to_test=N_DIFFMAP_COMPONENTS_TO_TEST,
                                      eigenvalue_floor=EIGENVALUE_FLOOR):
    """Pick the diffusion component that best follows the marker axis, weighted by eigenvalue."""
    diffmap = subset.obsm['X_diffmap']
    eigvals = np.asarray(subset.uns['diffmap_evals'], dtype=float)
    n_components = min(n_components_to_test, diffmap.shape[1], eigvals.size)

    rows = []
    for i in range(n_components):
        values = diffmap[:, i].astype(float)
        valid = np.isfinite(values) & np.isfinite(marker_axis)
        corr = (
            spearmanr(values[valid], marker_axis[valid]).correlation
            if valid.sum() >= MIN_VALID_FOR_SPEARMAN
            else np.nan
        )
        rows.append({
            'component': i,
            'eigenvalue': float(eigvals[i]),
            'spearman_with_marker_axis': corr,
            'score': (abs(corr) * eigvals[i]) if (np.isfinite(corr) and eigvals[i] >= eigenvalue_floor) else np.nan,
        })
    component_df = pd.DataFrame(rows)

    if component_df['score'].notna().any():
        best_component = int(component_df.loc[component_df['score'].idxmax(), 'component'])
    else:
        fallback = component_df['spearman_with_marker_axis'].abs().fillna(-1)
        best_component = int(component_df.loc[fallback.idxmax(), 'component'])

    return best_component, component_df


# ---------------------------------------------------------------------------
# PT-specific DPT
# ---------------------------------------------------------------------------

PT_SUBSET_DPT_SPEC = {
    'segment': 'PT',
    'marker_groups': PT_MARKER_GROUPS,
    'group_order': PT_GROUP_ORDER,
    'pseudotime_col': 'pt_subset_scanpy_dpt',
    'diffmap_col_template': 'pt_subset_diffmap_component_{i}',
}


def recompute_pt_subset_dpt(
    adata_obj,
    spec=PT_SUBSET_DPT_SPEC,
    n_neighbors_floor=15,
    random_state=0,
):
    segment = spec['segment']
    marker_groups = spec['marker_groups']
    group_order = spec['group_order']
    output_col = spec['pseudotime_col']
    diffmap_col_template = spec['diffmap_col_template']

    # ------- Subset to PT tubules -------
    subset_mask = adata_obj.obs['broad_tubule_marker_call'].astype(str).eq(segment).to_numpy()
    if subset_mask.sum() < 30:
        raise ValueError(f'{segment}: too few tubules for subset-specific DPT')
    subset = adata_obj[subset_mask].copy()

    # ------- Representation (harmonized PCA preferred; never UMAP) -------
    representation = choose_subset_representation(subset)

    # ------- Marker axis from segment-specific module scores -------
    available_groups, gene_order, gene_idx, missing = resolve_available_marker_groups(
        subset, marker_groups, group_order,
    )
    available_order = [group for group in group_order if group in available_groups]
    if len(available_order) < 2:
        raise ValueError(f'{segment}: at least two ordered marker modules are required')

    expression = subset.layers['lognorm'] if 'lognorm' in subset.layers else subset.X
    marker_expression = matrix_to_dense(expression[:, gene_idx]).T
    marker_z = robust_scale_rows(marker_expression)
    gene_position = {label: i for i, label in enumerate(gene_order)}

    early_group = available_order[0]   # e.g. PT_S1 markers (Slc5a1, Slc5a2, Slc22a6, ...)
    late_group = available_order[-1]   # e.g. PT_S3 markers (Miox, ...)
    early_idx = [gene_position[label] for label in available_groups[early_group]]
    late_idx = [gene_position[label] for label in available_groups[late_group]]
    early_score = np.nanmean(marker_z[early_idx], axis=0)
    late_score = np.nanmean(marker_z[late_idx], axis=0)
    marker_axis = late_score - early_score

    # ------- Neighbor graph (adaptive n_neighbors) -------
    n_neighbors_actual = max(n_neighbors_floor, int(np.sqrt(subset.n_obs)))
    n_neighbors_actual = min(n_neighbors_actual, subset.n_obs - 1)

    neighbors_key = f'{segment.lower()}_subset_dpt_neighbors'
    sc.pp.neighbors(
        subset,
        n_neighbors=n_neighbors_actual,
        use_rep=representation,
        key_added=neighbors_key,
        random_state=random_state,
    )
    sc.tl.diffmap(subset, neighbors_key=neighbors_key)

    # ------- Pick marker-aligned diffusion component (eigenvalue-weighted) -------
    best_component, component_df = choose_subset_diffusion_component(subset, marker_axis)
    best_eigval = float(component_df.loc[component_df['component'] == best_component, 'eigenvalue'].iloc[0])

    # Store the chosen component (signed coordinate; only sign-flipped, not rescaled).
    component_values = subset.obsm['X_diffmap'][:, best_component].astype(float)
    valid = np.isfinite(component_values) & np.isfinite(marker_axis)
    if valid.sum() >= MIN_VALID_FOR_SPEARMAN:
        corr = spearmanr(component_values[valid], marker_axis[valid]).correlation
        if np.isfinite(corr) and corr < 0:
            component_values = -component_values
    diffmap_col = diffmap_col_template.format(i=best_component)
    full_diffmap = np.full(adata_obj.n_obs, np.nan, dtype=float)
    full_diffmap[subset_mask] = component_values
    adata_obj.obs[diffmap_col] = full_diffmap

    # ------- Robust root (medoid of bottom-5% marker-axis tubules) -------
    root_index = choose_subset_root(marker_axis, subset, representation, bottom_quantile=0.05)
    subset.uns['iroot'] = root_index
    root_marker_value = float(marker_axis[root_index])
    root_obs_name = str(subset.obs_names[root_index])

    # ------- DPT -------
    sc.tl.dpt(subset, neighbors_key=neighbors_key)
    raw_dpt = subset.obs['dpt_pseudotime'].to_numpy(dtype=float)
    finite = np.isfinite(raw_dpt) & np.isfinite(marker_axis)
    if finite.sum() < 10:
        raise ValueError(f'{segment}: DPT produced too few finite values')

    scaled, raw_orientation_corr = orient_and_normalize(raw_dpt, marker_axis)

    # ------- Write back to full-AnnData -------
    full_values = np.full(adata_obj.n_obs, np.nan, dtype=float)
    full_values[subset_mask] = scaled
    adata_obj.obs[output_col] = full_values

    finite_scaled = np.isfinite(scaled)
    scaled_corr = (
        spearmanr(scaled[finite_scaled], marker_axis[finite_scaled]).correlation
        if finite_scaled.sum() >= MIN_VALID_FOR_SPEARMAN
        else np.nan
    )

    diagnostics = {
        'segment': segment,
        'n_tubules': int(subset.n_obs),
        'n_finite_dpt': int(finite_scaled.sum()),
        'representation': representation,
        'n_neighbors': int(n_neighbors_actual),
        'early_module': early_group,
        'late_module': late_group,
        'best_diffmap_component': int(best_component),
        'best_diffmap_eigenvalue': best_eigval,
        'root_obs_name': root_obs_name,
        'root_index_in_subset': int(root_index),
        'root_marker_axis': root_marker_value,
        'raw_dpt_marker_spearman': raw_orientation_corr,
        'scaled_dpt_marker_spearman': float(scaled_corr) if np.isfinite(scaled_corr) else np.nan,
        'missing_markers': '; '.join(missing),
        'pseudotime_col': output_col,
        'diffmap_col': diffmap_col,
    }
    return diagnostics, component_df


# ---------------------------------------------------------------------------
# Run for PT
# ---------------------------------------------------------------------------

pt_diagnostics, pt_component_df = recompute_pt_subset_dpt(adata)

print('\nPT diffusion component scan:')
display(pt_component_df)

pt_diagnostics_df = pd.DataFrame([pt_diagnostics])
pt_diagnostics_df.to_csv(
    OUTPUT_DIR / 'pt_subset_recomputed_dpt_diagnostics.csv',
    index=False,
)
display(pt_diagnostics_df)

> **Marker trend plot notes**
>
> Points are not evenly spaced by bin number. Cells are split into quantile bins along `total_scanpy_dpt`, so each bin contains roughly the same number of tubules. Each plotted point is placed at that bin's mean DPT value, which means points are closer together where many tubules occupy a narrow DPT range and farther apart where a bin spans a wider DPT interval.
>
> Error bars show the standard error of the mean across marker genes within the same marker panel at each DPT bin, after each gene's binned expression curve is min-max normalized. They summarize marker-to-marker variability within a panel, not uncertainty across individual tubules. Panels with one available marker gene have zero-width error bars.


### Marker-Trend Plotting Helpers

Defines the reusable marker-trend builders and plotting function used to validate each pseudospace method.


In [ ]:
def _resolve_marker_entry(entry, var_names_set, gene_lookup):
    """Given a {'label', 'candidates'} dict, return the first candidate present in the data, or None."""
    for candidate in entry.get('candidates', []):
        # 1) direct hit on var_names
        if candidate in var_names_set:
            return candidate
        # 2) case-insensitive lookup via gene_lookup
        resolved = gene_lookup.get(candidate.upper())
        if resolved is not None and resolved in var_names_set:
            return resolved
    return None


def _resolve_panel_to_gene_list(panel, var_names_set, gene_lookup):
    """Resolve a list of {'label','candidates'} entries into a flat list of gene symbols actually in the data."""
    resolved = []
    for entry in panel:
        if isinstance(entry, str):
            # tolerate plain-string entries too
            hit = entry if entry in var_names_set else gene_lookup.get(entry.upper())
            if hit is not None and hit in var_names_set:
                resolved.append(hit)
            continue
        if not isinstance(entry, dict):
            continue
        gene = _resolve_marker_entry(entry, var_names_set, gene_lookup)
        if gene is not None:
            resolved.append(gene)
    return resolved


def plot_pt_subset_marker_trend_figure(adata, pseudotime_col, method_label, output_name,
                                        segment_col='broad_tubule_marker_call'):
    """Two-panel figure: per-gene PT sub-segment trends + aggregated module trends, restricted to PT tubules."""
    pt_mask = adata.obs[segment_col].astype(str).eq('PT').to_numpy()
    finite_pt_mask = pt_mask & np.isfinite(adata.obs[pseudotime_col].to_numpy(dtype=float))
    if finite_pt_mask.sum() < 30:
        raise ValueError(f'Too few PT tubules with finite {pseudotime_col} for trend plotting')

    # Resolve PT marker panels into {module: [gene_symbol_in_data, ...]}, respecting PT_GROUP_ORDER.
    var_names_set = set(adata.var_names)
    pt_marker_panels = {}
    for group in PT_GROUP_ORDER:
        if group not in PT_MARKER_GROUPS:
            continue
        genes = _resolve_panel_to_gene_list(PT_MARKER_GROUPS[group], var_names_set, gene_lookup)
        if genes:
            pt_marker_panels[group] = genes
    if not pt_marker_panels:
        raise ValueError('No PT markers from PT_MARKER_GROUPS could be resolved against adata.var_names.')

    print('PT marker panels (resolved against data):')
    for group, genes in pt_marker_panels.items():
        print(f'  {group}: {genes}')

    marker_status, trend_df, peak_df, aggregate_df, aggregate_peak_df = _build_marker_trends_for_method(
        adata, pseudotime_col, pt_marker_panels, cell_mask=finite_pt_mask,
    )

    display(marker_status.assign(view='pt_sub_segments', method=method_label))
    display(
        peak_df.assign(view='pt_sub_segments', method=method_label)
              .sort_values(['marker_panel', 'peak_pseudotime', 'gene'])
    )

    panel_order = list(pt_marker_panels)
    # Use the project-defined PT module colors if available, else fall back to Set2.
    if all(group in PT_GROUP_COLORS for group in panel_order):
        palette = {group: PT_GROUP_COLORS[group] for group in panel_order}
    else:
        palette = dict(zip(panel_order, sns.color_palette('Set2', n_colors=len(panel_order))))

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.0), sharey=False)
    ax_gene, ax_module = axes

    # ----- Panel 1: individual PT marker genes, colored by sub-segment module -----
    for marker_panel in panel_order:
        panel_trend = trend_df.loc[trend_df['marker_panel'] == marker_panel]
        for gene, gene_trend in panel_trend.groupby('gene', observed=True):
            gene_trend = gene_trend.sort_values('pseudotime')
            ax_gene.plot(
                gene_trend['pseudotime'], gene_trend['normalized_expression'],
                color=palette[marker_panel], linewidth=1.6, alpha=0.55,
            )
    for marker_panel in panel_order:
        ax_gene.plot([], [], color=palette[marker_panel], linewidth=2.4, label=marker_panel)
    for _, row in peak_df.iterrows():
        ax_gene.scatter(
            row['peak_pseudotime'], row['normalized_expression'],
            color=palette[row['marker_panel']], edgecolor='black', linewidth=0.4, s=28, zorder=4, alpha=0.85,
        )
    ax_gene.set_title('Per-gene PT marker trends')
    ax_gene.set_xlabel(method_label)
    ax_gene.set_ylabel('Normalized expression')
    ax_gene.set_ylim(-0.05, 1.05)
    ax_gene.legend(title='PT module', fontsize=8, title_fontsize=9)

    # ----- Panel 2: aggregated module trends with peak markers -----
    for marker_panel in panel_order:
        panel_df = aggregate_df.loc[aggregate_df['marker_panel'] == marker_panel].sort_values('pseudotime')
        ax_module.errorbar(
            panel_df['pseudotime'], panel_df['normalized_expression'], yerr=panel_df['expression_sem'],
            color=palette[marker_panel], linewidth=2.8, marker='o', markersize=4,
            capsize=2, elinewidth=0.8, alpha=0.95, label=marker_panel,
        )
    for _, row in aggregate_peak_df.iterrows():
        ax_module.scatter(
            row['panel_peak_pseudotime'], row['normalized_expression'],
            color=palette[row['marker_panel']], edgecolor='black', linewidth=0.7, s=90, zorder=5,
        )
        ax_module.axvline(
            row['panel_peak_pseudotime'], color=palette[row['marker_panel']],
            linewidth=0.9, alpha=0.35,
        )
    peak_min = aggregate_peak_df['panel_peak_pseudotime'].min()
    peak_max = aggregate_peak_df['panel_peak_pseudotime'].max()
    pad = max(0.08, 0.18 * max(peak_max - peak_min, 1e-8))
    x_min = max(0.0, peak_min - pad)
    x_max = min(1.0, peak_max + pad)
    if x_max - x_min < 0.2:
        center = 0.5 * (x_min + x_max)
        x_min = max(0.0, center - 0.1)
        x_max = min(1.0, center + 0.1)
    ax_module.set_xlim(x_min, x_max)
    ax_module.set_ylim(-0.05, 1.05)
    ax_module.set_title('PT sub-segment module trends')
    ax_module.set_xlabel(method_label)
    ax_module.set_ylabel('Mean normalized expression')
    ax_module.legend(title='PT module', fontsize=8, title_fontsize=9)

    fig.suptitle(f'PT marker expression peaks along {method_label}', y=0.995, fontsize=14)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / output_name, dpi=200, bbox_inches='tight')
    plt.show()

    return {
        'marker_status': marker_status,
        'trend_df': trend_df,
        'peak_df': peak_df,
        'aggregate_df': aggregate_df,
        'aggregate_peak_df': aggregate_peak_df,
    }


pt_dpt_marker_trends = plot_pt_subset_marker_trend_figure(
    adata,
    pseudotime_col='pt_subset_scanpy_dpt',
    method_label='PT subset Scanpy DPT',
    output_name='pt_subset_scanpy_dpt_marker_trends.png',
)

print('PT marker-trend figure complete.')

In [ ]:
# Plot broad nephron and segment marker genes as one four-panel figure for a TOTAL pseudospace method.
def _expr_vector(adata, gene):
    expr_col = adata.X[:, adata.var_names.get_loc(gene)]
    return expr_col.toarray().ravel() if sparse.issparse(expr_col) else np.asarray(expr_col).ravel()


def _build_marker_trends_for_method(total_adata, pseudotime_col, marker_panels, n_bins=40, cell_mask=None):
    pt = total_adata.obs[pseudotime_col].to_numpy(dtype=float)
    valid_pt = np.isfinite(pt)
    if cell_mask is not None:
        valid_pt &= np.asarray(cell_mask, dtype=bool)
    if valid_pt.sum() < 30:
        raise ValueError(f'Too few finite {pseudotime_col} values to plot marker trends')

    bins = pd.qcut(pd.Series(pt[valid_pt]), q=n_bins, labels=False, duplicates='drop')
    valid_indices = np.where(valid_pt)[0]

    marker_rows = []
    for marker_panel, requested_genes in marker_panels.items():
        for requested_gene in requested_genes:
            gene = gene_lookup.get(requested_gene.upper())
            marker_rows.append({
                'marker_panel': marker_panel,
                'requested_gene': requested_gene,
                'gene': gene,
                'present': gene in total_adata.var_names if gene is not None else False,
            })
    marker_status = pd.DataFrame(marker_rows)
    present_marker_status = marker_status.loc[marker_status['present']].drop_duplicates(['marker_panel', 'gene']).copy()
    if present_marker_status.empty:
        raise ValueError('None of the selected marker genes are present after filtering')

    trend_rows = []
    for row in present_marker_status.to_dict('records'):
        gene = row['gene']
        expr = _expr_vector(total_adata, gene)
        tmp = pd.DataFrame({
            'bin': bins.to_numpy(),
            'pseudotime': pt[valid_indices],
            'expression': expr[valid_indices],
        })
        means = tmp.groupby('bin', observed=True).agg(
            pseudotime=('pseudotime', 'mean'),
            mean_expression=('expression', 'mean'),
        ).reset_index(drop=True)
        for mean_row in means.to_dict('records'):
            trend_rows.append({
                'marker_panel': row['marker_panel'],
                'gene': gene,
                'pseudotime': mean_row['pseudotime'],
                'mean_expression': mean_row['mean_expression'],
            })

    trend_df = pd.DataFrame(trend_rows)
    trend_df['normalized_expression'] = trend_df.groupby(['marker_panel', 'gene'], observed=True)['mean_expression'].transform(
        lambda x: (x - x.min()) / max(x.max() - x.min(), 1e-8)
    )
    peak_idx = trend_df.groupby(['marker_panel', 'gene'], observed=True)['normalized_expression'].idxmax()
    peak_df = trend_df.loc[peak_idx, ['marker_panel', 'gene', 'pseudotime', 'mean_expression', 'normalized_expression']].copy()
    peak_df = peak_df.rename(columns={'pseudotime': 'peak_pseudotime', 'mean_expression': 'peak_mean_expression'})
    peak_df = peak_df.sort_values(['marker_panel', 'peak_pseudotime', 'gene']).reset_index(drop=True)
    aggregate_df = trend_df.groupby(['marker_panel', 'pseudotime'], observed=True, as_index=False).agg(
        normalized_expression=('normalized_expression', 'mean'),
        expression_sem=('normalized_expression', lambda x: x.sem() if len(x) > 1 else 0.0),
    )
    aggregate_peak_idx = aggregate_df.groupby('marker_panel', observed=True)['normalized_expression'].idxmax()
    aggregate_peak_df = aggregate_df.loc[aggregate_peak_idx].rename(columns={'pseudotime': 'panel_peak_pseudotime'})
    return marker_status, trend_df, peak_df, aggregate_df, aggregate_peak_df


def plot_total_marker_trend_figure(total_adata, pseudotime_col, method_label, output_name):
    broad_marker_panels = {
        'PT': BROAD_SEGMENT_MARKERS['PT'],
        'TAL': BROAD_SEGMENT_MARKERS['TAL'],
        'DCT': BROAD_SEGMENT_MARKERS['DCT'],
        'CD': BROAD_SEGMENT_MARKERS['CNT_CD'],
    }
    zoom_marker_panels = {
        'PT': CONTINUUM_CONFIGS['PT']['segment_markers'],
        'DCT': CONTINUUM_CONFIGS['DCT']['segment_markers'],
        'TAL': CONTINUUM_CONFIGS['TAL']['segment_markers'],
    }

    broad_status, _, broad_peak_df, broad_aggregate_df, broad_aggregate_peak_df = _build_marker_trends_for_method(
        total_adata, pseudotime_col, broad_marker_panels
    )
    zoom_results = {
        region: _build_marker_trends_for_method(
            total_adata,
            pseudotime_col,
            panels,
            cell_mask=total_adata.obs['total_segment_marker_call'].eq(region).to_numpy(),
        )
        for region, panels in zoom_marker_panels.items()
    }

    display(broad_status.assign(view='broad_nephron', method=method_label))
    for nephron_region, (status, _, peak_df, _, _) in zoom_results.items():
        display(status.assign(view=f'{nephron_region}_zoom', method=method_label))
        display(peak_df.assign(view=f'{nephron_region}_zoom', method=method_label).sort_values(['marker_panel', 'peak_pseudotime', 'gene']))

    fig, axes = plt.subplots(2, 2, figsize=(14, 8.5), sharey=False)
    ax_broad, ax_pt, ax_dct, ax_tal = axes.ravel()

    broad_order = list(broad_marker_panels)
    broad_palette = dict(zip(broad_order, sns.color_palette('Dark2', n_colors=len(broad_order))))
    zoom_palettes = {
        region: dict(zip(zoom_marker_panels[region], sns.color_palette('Set2', n_colors=len(zoom_marker_panels[region]))))
        for region in zoom_marker_panels
    }

    for marker_panel in broad_order:
        panel_df = broad_aggregate_df.loc[broad_aggregate_df['marker_panel'] == marker_panel].sort_values('pseudotime')
        ax_broad.errorbar(
            panel_df['pseudotime'], panel_df['normalized_expression'], yerr=panel_df['expression_sem'],
            color=broad_palette[marker_panel], linewidth=2.6, marker='o', markersize=4,
            capsize=2, elinewidth=0.8, alpha=0.95, label=marker_panel,
        )
    for _, row in broad_aggregate_peak_df.iterrows():
        ax_broad.scatter(row['panel_peak_pseudotime'], row['normalized_expression'], color=broad_palette[row['marker_panel']], edgecolor='black', linewidth=0.7, s=90, zorder=5)
        ax_broad.axvline(row['panel_peak_pseudotime'], color=broad_palette[row['marker_panel']], linewidth=0.9, alpha=0.3)
    ax_broad.set_title('Total nephron marker panels')
    ax_broad.set_xlabel(method_label)
    ax_broad.set_ylabel('Mean normalized expression')
    ax_broad.legend(title='Panel', fontsize=8, title_fontsize=9)

    for nephron_region, ax in [('PT', ax_pt), ('DCT', ax_dct), ('TAL', ax_tal)]:
        _, _, peak_df, aggregate_df, aggregate_peak_df = zoom_results[nephron_region]
        panel_order = list(zoom_marker_panels[nephron_region])
        palette = zoom_palettes[nephron_region]
        for marker_panel in panel_order:
            panel_df = aggregate_df.loc[aggregate_df['marker_panel'] == marker_panel].sort_values('pseudotime')
            ax.errorbar(
                panel_df['pseudotime'], panel_df['normalized_expression'], yerr=panel_df['expression_sem'],
                color=palette[marker_panel], linewidth=2.8, marker='o', markersize=4,
                capsize=2, elinewidth=0.8, alpha=0.95, label=marker_panel,
            )
        for _, row in aggregate_peak_df.iterrows():
            ax.scatter(row['panel_peak_pseudotime'], row['normalized_expression'], color=palette[row['marker_panel']], edgecolor='black', linewidth=0.6, s=80, zorder=5)
            ax.axvline(row['panel_peak_pseudotime'], color=palette[row['marker_panel']], linewidth=0.9, alpha=0.35)
        peak_min = peak_df['peak_pseudotime'].min()
        peak_max = peak_df['peak_pseudotime'].max()
        pad = max(0.08, 0.18 * max(peak_max - peak_min, 1e-8))
        x_min = max(0, peak_min - pad)
        x_max = min(1, peak_max + pad)
        if x_max - x_min < 0.2:
            center = 0.5 * (x_min + x_max)
            x_min = max(0, center - 0.1)
            x_max = min(1, center + 0.1)
        ax.set_xlim(x_min, x_max)
        ax.set_ylim(-0.05, 1.05)
        ax.set_title(f'{nephron_region} segment marker zoom')
        ax.set_xlabel(method_label)
        ax.set_ylabel('Normalized expression')
        ax.legend(title='Panel', fontsize=8, title_fontsize=9)

    fig.suptitle(f'Marker expression peaks along {method_label}', y=0.995, fontsize=14)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / output_name, dpi=200, bbox_inches='tight')
    plt.show()
    return {
        'broad': (broad_status, broad_peak_df, broad_aggregate_df),
        'zoom': zoom_results,
    }


In [ ]:

dpt_marker_trends = plot_total_marker_trend_figure(
    dpt_loaded['TOTAL']['adata'],
    'total_scanpy_dpt',
    'TOTAL Scanpy DPT',
    'total_scanpy_dpt_marker_trends.png',
)

print('Marker-trend helper functions are defined.')

## Method 2: Slingshot


In [ ]:
# ---------------------------------------------------------------------
# One-time R/Bioconductor setup for Slingshot through rpy2
# Installs packages into a persistent local folder: ./r_libs
# ---------------------------------------------------------------------
from pathlib import Path
import os
import rpy2.robjects as ro
from rpy2.robjects.packages import importr, isinstalled

# Try to use PROJECT_DIR if your notebook already has it; otherwise use current directory.
try:
    project_dir = Path(PROJECT_DIR).resolve()
except NameError:
    project_dir = Path.cwd().resolve()

# Persistent local R package directory.
R_LIB_DIR = project_dir / "r_libs"
R_LIB_DIR.mkdir(parents=True, exist_ok=True)

local_r_lib = str(R_LIB_DIR)

# Make R use this library path first.
ro.r.assign("local_r_lib", local_r_lib)
ro.r("""
dir.create(local_r_lib, recursive = TRUE, showWarnings = FALSE)
.libPaths(c(local_r_lib, .libPaths()))
message("Using R library paths:")
print(.libPaths())
""")

utils = importr("utils")

# Install BiocManager if missing.
if not isinstalled("BiocManager", lib_loc=local_r_lib):
    print(f"Installing BiocManager into {local_r_lib}")
    utils.install_packages(
        ro.StrVector(["BiocManager"]),
        repos="https://cloud.r-project.org",
        lib=local_r_lib
    )

biocmanager = importr("BiocManager", lib_loc=local_r_lib)

# CRAN packages needed by Slingshot dependencies.
cran_pkgs = [
    "Matrix",
    "Rcpp",
    "RcppArmadillo",
    "matrixStats",
    "igraph",
    "princurve",
]

# Bioconductor packages needed for the SingleCellExperiment + Slingshot runtime dependency chain.
bioc_pkgs = [
    "BiocGenerics",
    "S4Vectors",
    "IRanges",
    "GenomeInfoDb",
    "XVector",
    "GenomicRanges",
    "MatrixGenerics",
    "S4Arrays",
    "SparseArray",
    "DelayedArray",
    "DelayedMatrixStats",
    "Biobase",
    "SummarizedExperiment",
    "SingleCellExperiment",
    "TrajectoryUtils",
    "slingshot",
]

# Install missing CRAN packages.
missing_cran = [pkg for pkg in cran_pkgs if not isinstalled(pkg, lib_loc=local_r_lib)]
if missing_cran:
    print(f"Installing missing CRAN packages: {missing_cran}")
    utils.install_packages(
        ro.StrVector(missing_cran),
        repos="https://cloud.r-project.org",
        lib=local_r_lib
    )

# Install missing Bioconductor packages.
missing_bioc = [pkg for pkg in bioc_pkgs if not isinstalled(pkg, lib_loc=local_r_lib)]
if missing_bioc:
    print(f"Installing missing Bioconductor packages: {missing_bioc}")
    ro.r("options(repos = BiocManager::repositories())")
    biocmanager.install(
        ro.StrVector(missing_bioc),
        ask=False,
        update=False,
        dependencies=ro.StrVector(["Depends", "Imports", "LinkingTo"]),
        lib=local_r_lib
    )

# Verify core packages and known runtime dependencies load.
required_pkgs = [
    "Matrix",
    "matrixStats",
    "princurve",
    "BiocGenerics",
    "S4Vectors",
    "IRanges",
    "GenomeInfoDb",
    "GenomicRanges",
    "MatrixGenerics",
    "S4Arrays",
    "SparseArray",
    "DelayedArray",
    "DelayedMatrixStats",
    "Biobase",
    "SummarizedExperiment",
    "SingleCellExperiment",
    "TrajectoryUtils",
    "slingshot",
]
for pkg in required_pkgs:
    print(f"Checking R package: {pkg}")
    importr(pkg, lib_loc=local_r_lib, on_conflict="warn")

print("✅ Slingshot R environment is installed and ready.")
print(f"Persistent R library directory: {local_r_lib}")

In [ ]:
slingshot_loaded = clone_preprocessed_continua()

# Prepare the optional R bridge and Slingshot dependencies.
r_ready = False
slingshot_available = False
try:
    import rpy2.robjects as ro
    from rpy2.robjects import numpy2ri, pandas2ri
    from rpy2.robjects.conversion import localconverter
    from rpy2.robjects.packages import importr, isinstalled

    local_r_lib = str(R_LIB_DIR)
    ro.r.assign('local_r_lib', local_r_lib)
    ro.r('dir.create(local_r_lib, recursive=TRUE, showWarnings=FALSE)')
    ro.r('.libPaths(c(local_r_lib, .libPaths()))')

    # Install BiocManager locally when needed so Bioconductor packages can be resolved.
    if not isinstalled('BiocManager', lib_loc=local_r_lib):
        print(f'Installing BiocManager into {local_r_lib}')
        utils = importr('utils')
        utils.install_packages(ro.StrVector(['BiocManager']), repos='https://cloud.r-project.org', lib=local_r_lib)

    # Install missing Slingshot dependencies into the project-local R library.
    if isinstalled('BiocManager', lib_loc=local_r_lib):
        missing = [pkg for pkg in ['S4Vectors', 'SingleCellExperiment', 'slingshot'] if not isinstalled(pkg, lib_loc=local_r_lib)]
        if missing:
            print(f'Attempting Bioconductor installation into {local_r_lib}: {missing}')
            ro.r('options(repos = c(CRAN = "https://cloud.r-project.org"))')
            try:
                importr('BiocManager', lib_loc=local_r_lib).install(ro.StrVector(missing), ask=False, update=False, lib=local_r_lib)
            except Exception as exc:
                print(f'Optional Slingshot package install did not fully succeed: {exc}')

    importr('Matrix')
    importr('S4Vectors')
    importr('SingleCellExperiment')
    importr('slingshot')
    r_ready = True
    slingshot_available = True
except Exception as exc:
    print(f'Slingshot unavailable: {exc}')

# Run Slingshot once on the total nephron continuum.
for continuum, obj in {'TOTAL': slingshot_loaded['TOTAL']}.items():
    print('\n' + '=' * 80)
    print(f'Running Slingshot for {continuum}')
    adata = obj['adata']
    config = obj['config']
    prefix = config['prefix']
    marker_axis = adata.obs[f'{prefix}_marker_axis'].to_numpy(dtype=float)

    if not (r_ready and slingshot_available):
        print('Skipped Slingshot because required R packages are unavailable')
        continue

    # Slingshot receives an unsupervised Leiden clustering, not marker-derived labels.
    rep = next((key for key in [f'X_pca_harmony_{prefix}', 'X_pca_harmony', 'X_harmony', 'X_pca', f'X_umap_harmony_{prefix}', 'X_umap'] if key in adata.obsm), None)
    if rep is None:
        sc.pp.pca(adata, n_comps=min(N_PCS_FALLBACK, adata.n_vars - 1), random_state=RANDOM_STATE)
        rep = 'X_pca'
    print(f'{prefix}: Slingshot/Leiden uses representation {rep}')

    neighbors_key = f'{prefix}_slingshot_neighbors'
    leiden_col = f'{prefix}_slingshot_leiden'
    sc.pp.neighbors(adata, n_neighbors=N_NEIGHBORS, use_rep=rep, key_added=neighbors_key, random_state=RANDOM_STATE)
    sc.tl.leiden(adata, resolution=1.0, neighbors_key=neighbors_key, key_added=leiden_col, random_state=RANDOM_STATE)

    # Pick the starting cluster only after unsupervised clustering, using the marker-axis score.
    cluster_marker_means = adata.obs.assign(_marker_axis=marker_axis).groupby(leiden_col, observed=True)['_marker_axis'].mean()
    start_cluster = str(cluster_marker_means.idxmin())
    print(f'{prefix}: Leiden clusters={adata.obs[leiden_col].nunique()}, start cluster={start_cluster}')
    display(cluster_marker_means.sort_values().to_frame('mean_marker_axis'))

    # Run Slingshot on every tubule in the total nephron continuum.
    sample_idx = np.arange(adata.n_obs)

    sampled = adata[sample_idx].copy()
    embedding = np.asarray(sampled.obsm[rep], dtype=float)
    if embedding.shape[1] > 10:
        embedding = embedding[:, :10]
    clusters = sampled.obs[leiden_col].astype(str).to_numpy()
    if start_cluster not in set(clusters):
        sampled_cluster_means = sampled.obs.assign(_marker_axis=marker_axis[sample_idx]).groupby(leiden_col, observed=True)['_marker_axis'].mean()
        start_cluster = str(sampled_cluster_means.idxmin())
        print(f'{prefix}: start cluster was not sampled; using sampled start cluster={start_cluster}')

    # Build a minimal SingleCellExperiment in R and run Slingshot on embedding + Leiden labels + start cluster.
    try:
        with localconverter(ro.default_converter + numpy2ri.converter + pandas2ri.converter):
            r_emb = ro.r.matrix(embedding, nrow=embedding.shape[0], ncol=embedding.shape[1])
            r_clusters = ro.StrVector(clusters)
            ro.r.assign('emb_tmp', r_emb)
            ro.r.assign('clusters_tmp', r_clusters)
            ro.r.assign('start_cluster_tmp', start_cluster)
            ro.r('counts_tmp <- Matrix::Matrix(0, nrow=1, ncol=nrow(emb_tmp), sparse=TRUE)')
            ro.r('rownames(counts_tmp) <- "dummy"')
            ro.r('colnames(counts_tmp) <- paste0("cell_", seq_len(ncol(counts_tmp)))')
            ro.r('sce_tmp <- SingleCellExperiment::SingleCellExperiment(assays=list(counts=counts_tmp))')
            ro.r('reducedDims(sce_tmp) <- S4Vectors::SimpleList(TRAJ=emb_tmp)')
            ro.r('colLabels(sce_tmp) <- clusters_tmp')
            ro.r('sce_tmp <- slingshot::slingshot(sce_tmp, clusterLabels=colLabels(sce_tmp), reducedDim="TRAJ", start.clus=start_cluster_tmp)')
            pt = np.asarray(ro.r('slingPseudotime(sce_tmp, na=FALSE)[, 1]'), dtype=float)
    except Exception as exc:
        print(f'Slingshot failed for {continuum}: {exc}')
        continue

    # Orient and normalize Slingshot pseudotime before inserting it back into the full object.
    valid = np.isfinite(pt) & np.isfinite(marker_axis[sample_idx])
    oriented = pt.copy()
    if valid.sum() >= 10:
        corr = spearmanr(oriented[valid], marker_axis[sample_idx][valid]).correlation
        if np.isfinite(corr) and corr < 0:
            oriented = -oriented
    finite = np.isfinite(oriented)
    normalized_sample = np.full(oriented.shape, np.nan, dtype=float)
    if finite.sum() > 0:
        p5, p95 = np.nanpercentile(oriented[finite], [5, 95])
        normalized_sample[finite] = np.clip((oriented[finite] - p5) / max(p95 - p5, 1e-8), 0, 1)

    # Insert Slingshot pseudotime back into the full object.
    full = np.full(adata.n_obs, np.nan, dtype=float)
    full[sample_idx] = normalized_sample
    col = f'{prefix}_slingshot'
    adata.obs[col] = full
    obj['method_cols'].append(col)
    obj['method_notes'][col] = f'Slingshot ran on {len(sample_idx):,}/{adata.n_obs:,} cells using {rep}, Leiden labels, start cluster {start_cluster}'


In [ ]:
# Plot Slingshot marker trends using the same four-panel layout.
slingshot_marker_trends = plot_total_marker_trend_figure(
    slingshot_loaded['TOTAL']['adata'],
    'total_slingshot',
    'TOTAL Slingshot',
    'total_slingshot_marker_trends.png',
)


## Method 3: Monocle


In [ ]:
# ---------------------------------------------------------------------
# One-time R/GitHub setup for Monocle3 through rpy2
# Monocle3 is installed from GitHub because it is not available from Bioconductor 3.20.
# Existing packages are reused; only missing packages are installed.
# ---------------------------------------------------------------------
from pathlib import Path
import os
import shutil
import subprocess
import sys
import rpy2.robjects as ro
from rpy2.robjects.packages import importr, isinstalled

try:
    project_dir = Path(PROJECT_DIR).resolve()
except NameError:
    project_dir = Path.cwd().resolve()

R_LIB_DIR = project_dir / "r_libs"
R_LIB_DIR.mkdir(parents=True, exist_ok=True)
local_r_lib = str(R_LIB_DIR)

# nloptr/lme4 and s2/sf compile from source on Linux; make cmake available from this Python env if needed.
if shutil.which("cmake") is None:
    print("cmake was not found on PATH; installing Python cmake wheel for R package compilation")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "cmake"])
python_bin = str(Path(sys.executable).resolve().parent)
os.environ["PATH"] = python_bin + os.pathsep + os.environ.get("PATH", "")
os.environ["USE_BUNDLED_LIBUV"] = "1"

ro.r.assign("local_r_lib", local_r_lib)
ro.r.assign("python_bin", python_bin)
ro.r("""
dir.create(local_r_lib, recursive = TRUE, showWarnings = FALSE)
.libPaths(c(local_r_lib, .libPaths()))
Sys.setenv(PATH = paste(python_bin, Sys.getenv("PATH"), sep = .Platform$path.sep))
Sys.setenv(USE_BUNDLED_LIBUV = "1")
message("Using R library paths:")
print(.libPaths())
message("Using cmake: ", Sys.which("cmake"))
""")

utils = importr("utils")

if not isinstalled("BiocManager", lib_loc=local_r_lib):
    print(f"Installing BiocManager into {local_r_lib}")
    utils.install_packages(
        ro.StrVector(["BiocManager"]),
        repos="https://cloud.r-project.org",
        lib=local_r_lib,
    )

if not isinstalled("remotes", lib_loc=local_r_lib):
    print(f"Installing remotes into {local_r_lib}")
    utils.install_packages(
        ro.StrVector(["remotes"]),
        repos="https://cloud.r-project.org",
        lib=local_r_lib,
    )

biocmanager = importr("BiocManager", lib_loc=local_r_lib)
remotes = importr("remotes", lib_loc=local_r_lib)

# CRAN dependencies that Monocle3 requires directly or through lme4/plotly/sf/shiny/spdep.
# Install in stages so downstream packages retry after low-level compiled deps succeed.
monocle_cran_groups = [
    ["fs", "sass"],
    ["bslib"],
    ["rmarkdown", "shiny"],
    ["htmlwidgets"],
    ["plotly"],
    ["nloptr", "lme4"],
    ["s2", "sf", "spdep"],
]
monocle_cran_pkgs = [pkg for group in monocle_cran_groups for pkg in group]
for cran_group in monocle_cran_groups:
    missing_group = [pkg for pkg in cran_group if not isinstalled(pkg, lib_loc=local_r_lib)]
    if missing_group:
        print(f"Installing missing Monocle3 CRAN dependencies: {missing_group}")
        utils.install_packages(
            ro.StrVector(missing_group),
            repos="https://cloud.r-project.org",
            lib=local_r_lib,
            dependencies=ro.StrVector(["Depends", "Imports", "LinkingTo"]),
        )
        still_missing_group = [pkg for pkg in missing_group if not isinstalled(pkg, lib_loc=local_r_lib)]
        if still_missing_group:
            raise RuntimeError(
                "These Monocle3 CRAN dependencies failed to install: "
                + ", ".join(still_missing_group)
                + ". Check the R install output immediately above for the first package error."
            )

# Required by current Monocle3 GitHub install instructions and runtime imports.
monocle_bioc_pkgs = [
    "BiocGenerics",
    "DelayedArray",
    "DelayedMatrixStats",
    "limma",
    "S4Vectors",
    "SingleCellExperiment",
    "SummarizedExperiment",
    "batchelor",
    "HDF5Array",
    "ggrastr",
]
missing_bioc = [pkg for pkg in monocle_bioc_pkgs if not isinstalled(pkg, lib_loc=local_r_lib)]
if missing_bioc:
    print(f"Installing missing Monocle3 Bioconductor dependencies: {missing_bioc}")
    ro.r("options(repos = BiocManager::repositories())")
    biocmanager.install(
        ro.StrVector(missing_bioc),
        ask=False,
        update=False,
        dependencies=ro.StrVector(["Depends", "Imports", "LinkingTo"]),
        lib=local_r_lib,
    )

# BPCells is required by the current Monocle3 GitHub package.
if not isinstalled("BPCells", lib_loc=local_r_lib):
    print(f"Installing BPCells from GitHub into {local_r_lib}")
    remotes.install_github(
        "bnprks/BPCells/r",
        lib=local_r_lib,
        upgrade="never",
        dependencies=ro.StrVector(["Depends", "Imports", "LinkingTo"]),
    )

required_before_monocle = monocle_cran_pkgs + monocle_bioc_pkgs + ["BPCells"]
still_missing = [pkg for pkg in required_before_monocle if not isinstalled(pkg, lib_loc=local_r_lib)]
if still_missing:
    raise RuntimeError(
        "These Monocle3 dependencies are still missing after installation: "
        + ", ".join(still_missing)
        + ". Check the R install output above for the first package that failed."
    )

if not isinstalled("monocle3", lib_loc=local_r_lib):
    print(f"Installing monocle3 from GitHub into {local_r_lib}")
    remotes.install_github(
        "cole-trapnell-lab/monocle3",
        lib=local_r_lib,
        upgrade="never",
        dependencies=ro.StrVector(["Depends", "Imports", "LinkingTo"]),
    )

print("Checking R package: monocle3")
importr("monocle3", lib_loc=local_r_lib, on_conflict="warn")
print("✅ Monocle3 R environment is installed and ready.")
print(f"Persistent R library directory: {local_r_lib}")


In [ ]:
monocle_loaded = clone_preprocessed_continua()
for obj in monocle_loaded.values():
    obj['gene_lookup'] = gene_lookup

# Prepare an independent R bridge and Monocle3 dependencies.
r_ready = False
try:
    import rpy2.robjects as ro
    from rpy2.robjects import numpy2ri, pandas2ri
    from rpy2.robjects.conversion import localconverter
    from rpy2.robjects.packages import importr, isinstalled

    local_r_lib = str(R_LIB_DIR)
    ro.r.assign('local_r_lib', local_r_lib)
    ro.r('dir.create(local_r_lib, recursive=TRUE, showWarnings=FALSE)')
    ro.r('.libPaths(c(local_r_lib, .libPaths()))')
    if not isinstalled('BiocManager', lib_loc=local_r_lib):
        print(f'Installing BiocManager into {local_r_lib}')
        utils = importr('utils')
        utils.install_packages(ro.StrVector(['BiocManager']), repos='https://cloud.r-project.org', lib=local_r_lib)
    r_ready = True
except Exception as exc:
    print(f'R bridge unavailable for Monocle3: {exc}')

monocle_available = False
if r_ready:
    try:
        importr('monocle3', lib_loc=str(R_LIB_DIR), on_conflict='warn')
        importr('SingleCellExperiment', lib_loc=str(R_LIB_DIR), on_conflict='warn')
        ro.r('suppressPackageStartupMessages(library(SingleCellExperiment))')
        monocle_available = True
    except Exception as exc:
        print(f'Monocle3 unavailable: {exc}')
        print('Run the Monocle3 setup cell immediately above this one, then rerun this cell.')
else:
    print('Monocle3 skipped because rpy2/R setup is unavailable')


def _run_monocle3_total(loaded, *, use_harmonized_umap=False):
    """Run Monocle3 on TOTAL, optionally learning the graph on the existing harmonized UMAP."""
    label = 'Monocle3 on harmonized UMAP' if use_harmonized_umap else 'Monocle3'

    for continuum, obj in {'TOTAL': loaded['TOTAL']}.items():
        print('\n' + '=' * 80)
        print(f'Running {label} for {continuum}')
        adata = obj['adata']
        config = obj['config']
        prefix = config['prefix']
        marker_axis = adata.obs[f'{prefix}_marker_axis'].to_numpy(dtype=float)

        if not monocle_available:
            print(f'Skipped {label} because required R packages are unavailable')
            continue

        # Run Monocle3 on every tubule in the total nephron continuum.
        sample_idx = np.arange(adata.n_obs)

        # Monocle3 expects counts and performs its own normalization/PCA. For the harmonized-UMAP
        # variant, that preprocessing is still used to construct the CDS, but graph learning uses
        # the existing harmonized UMAP coordinates instead of Monocle3's recomputed UMAP.
        sampled = adata[sample_idx].copy()
        count_matrix = sampled.layers['counts'] if 'counts' in sampled.layers else sampled.X
        count_matrix = sparse.csr_matrix(count_matrix)
        nonzero_expression_cells = np.asarray(count_matrix.sum(axis=1)).ravel() > 0
        if nonzero_expression_cells.sum() < 30:
            print(f'Skipped {label} because too few cells have nonzero counts in the filtered feature matrix')
            continue
        if nonzero_expression_cells.sum() < len(nonzero_expression_cells):
            print(f'{prefix}: dropping {len(nonzero_expression_cells) - int(nonzero_expression_cells.sum()):,} cells with zero filtered-feature counts before {label}')
        monocle_sample_idx = sample_idx[nonzero_expression_cells]
        sampled_for_monocle = sampled[nonzero_expression_cells].copy()
        counts_genes_by_cells = sparse.csc_matrix(count_matrix[nonzero_expression_cells].T)
        genes = sampled_for_monocle.var_names.astype(str).tolist()
        metadata_cols = [col for col in ['sample', 'condition', 'species', 'broad_tubule_marker_call'] if col in sampled_for_monocle.obs.columns]
        cell_metadata = sampled_for_monocle.obs[metadata_cols].copy()
        cell_metadata['cell_id'] = sampled_for_monocle.obs_names
        cell_metadata.index = sampled_for_monocle.obs_names
        gene_metadata = pd.DataFrame({'gene_short_name': genes}, index=genes)

        umap_key = None
        umap_df = None
        if use_harmonized_umap:
            preferred_umap_keys = [f'X_umap_harmony_{prefix}', 'X_umap_harmony', 'X_umap']
            umap_key = next((key for key in preferred_umap_keys if key in sampled_for_monocle.obsm), None)
            if umap_key is None:
                print(f'Skipped {label} because no harmonized UMAP was found in {preferred_umap_keys}')
                continue
            umap = np.asarray(sampled_for_monocle.obsm[umap_key])[:, :2]
            finite_umap = np.isfinite(umap).all(axis=1)
            if finite_umap.sum() < 30:
                print(f'Skipped {label} because {umap_key} has too few finite coordinates')
                continue
            if finite_umap.sum() < len(finite_umap):
                print(f'{prefix}: dropping {len(finite_umap) - int(finite_umap.sum()):,} cells with non-finite {umap_key} before {label}')
                monocle_sample_idx = monocle_sample_idx[finite_umap]
                sampled_for_monocle = sampled_for_monocle[finite_umap].copy()
                counts_genes_by_cells = sparse.csc_matrix(count_matrix[monocle_sample_idx].T)
                cell_metadata = cell_metadata.loc[sampled_for_monocle.obs_names].copy()
                umap = np.asarray(sampled_for_monocle.obsm[umap_key])[:, :2]
            umap_df = pd.DataFrame(umap, index=sampled_for_monocle.obs_names, columns=['UMAP_1', 'UMAP_2'])

        # Build a sparse genes-by-cells count matrix in R, then learn and root the Monocle3 graph.
        try:
            with localconverter(ro.default_converter + numpy2ri.converter + pandas2ri.converter):
                ro.r.assign('counts_i_tmp', ro.IntVector(counts_genes_by_cells.indices))
                ro.r.assign('counts_p_tmp', ro.IntVector(counts_genes_by_cells.indptr))
                ro.r.assign('counts_x_tmp', ro.FloatVector(counts_genes_by_cells.data))
                ro.r.assign('counts_dims_tmp', ro.IntVector(counts_genes_by_cells.shape))
                ro.r.assign('cell_metadata_tmp', cell_metadata)
                ro.r.assign('gene_metadata_tmp', gene_metadata)
                ro.r('expr_tmp <- Matrix::sparseMatrix(i=counts_i_tmp, p=counts_p_tmp, x=counts_x_tmp, dims=counts_dims_tmp, index1=FALSE, giveCsparse=TRUE)')
                ro.r('rownames(expr_tmp) <- rownames(gene_metadata_tmp)')
                ro.r('colnames(expr_tmp) <- rownames(cell_metadata_tmp)')
                ro.r('cds_tmp <- monocle3::new_cell_data_set(expr_tmp, cell_metadata=cell_metadata_tmp, gene_metadata=gene_metadata_tmp)')
                ro.r('num_dim_tmp <- min(50, nrow(expr_tmp) - 1, ncol(expr_tmp) - 1)')
                ro.r('cds_tmp <- monocle3::preprocess_cds(cds_tmp, num_dim=num_dim_tmp, norm_method="log")')
                if not use_harmonized_umap and 'sample' in cell_metadata.columns and cell_metadata['sample'].nunique(dropna=True) > 1:
                    print(f'{prefix}: aligning Monocle3 CDS by sample')
                    ro.r('cds_tmp <- monocle3::align_cds(cds_tmp, alignment_group="sample")')
                elif use_harmonized_umap:
                    print(f'{prefix}: using existing {umap_key} coordinates for Monocle3 graph learning')
                else:
                    print(f'{prefix}: skipping Monocle3 sample alignment because fewer than two samples are present')

                if use_harmonized_umap:
                    ro.r.assign('umap_tmp', umap_df)
                    ro.r('reducedDims(cds_tmp)$UMAP <- as.matrix(umap_tmp[colnames(cds_tmp), ])')
                else:
                    ro.r('cds_tmp <- monocle3::reduce_dimension(cds_tmp, reduction_method="UMAP", preprocess_method=if ("Aligned" %in% reducedDimNames(cds_tmp)) "Aligned" else "PCA")')
                ro.r('cds_tmp <- monocle3::cluster_cells(cds_tmp, reduction_method="UMAP")')
                ro.r('cds_tmp <- monocle3::learn_graph(cds_tmp, use_partition=FALSE)')
                if 'sample' in sampled_for_monocle.obs.columns:
                    root_cells = []
                    for _, sample_positions in sampled_for_monocle.obs.groupby('sample', observed=True).indices.items():
                        sample_positions = np.asarray(sample_positions)
                        sample_marker_axis = marker_axis[monocle_sample_idx[sample_positions]]
                        finite_positions = sample_positions[np.isfinite(sample_marker_axis)]
                        if len(finite_positions):
                            best_position = finite_positions[np.nanargmin(marker_axis[monocle_sample_idx[finite_positions]])]
                            root_cells.append(sampled_for_monocle.obs_names[best_position])
                else:
                    root_cells = [sampled_for_monocle.obs_names[int(np.nanargmin(marker_axis[monocle_sample_idx]))]]
                print(f'{prefix}: ordering {label} from {len(root_cells)} early root cells')
                ro.r.assign('root_cells_tmp', ro.StrVector(root_cells))
                ro.r('cds_tmp <- monocle3::order_cells(cds_tmp, root_cells=root_cells_tmp)')
                pt = np.asarray(ro.r('monocle3::pseudotime(cds_tmp)'), dtype=float)
        except Exception as exc:
            print(f'{label} failed for {continuum}: {exc}')
            continue

        # Orient and normalize Monocle pseudotime before inserting it back into the full object.
        valid = np.isfinite(pt) & np.isfinite(marker_axis[monocle_sample_idx])
        oriented = pt.copy()
        if valid.sum() >= 10:
            corr = spearmanr(oriented[valid], marker_axis[monocle_sample_idx][valid]).correlation
            if np.isfinite(corr) and corr < 0:
                oriented = -oriented
        finite = np.isfinite(oriented)
        normalized_sample = np.full(oriented.shape, np.nan, dtype=float)
        if finite.sum() > 0:
            p5, p95 = np.nanpercentile(oriented[finite], [5, 95])
            normalized_sample[finite] = np.clip((oriented[finite] - p5) / max(p95 - p5, 1e-8), 0, 1)

        # Cells removed for zero expression or missing UMAP coordinates remain NaN.
        full = np.full(adata.n_obs, np.nan, dtype=float)
        full[monocle_sample_idx] = normalized_sample
        col = f'{prefix}_monocle3_harmony_umap' if use_harmonized_umap else f'{prefix}_monocle3'
        adata.obs[col] = full
        obj['method_cols'].append(col)
        if use_harmonized_umap:
            obj['method_notes'][col] = f'Monocle3 ran on {len(monocle_sample_idx):,}/{adata.n_obs:,} cells using sparse raw counts, native preprocessing, existing {umap_key} coordinates for UMAP-space clustering/graph learning, and one early root cell per sample'
        else:
            obj['method_notes'][col] = f'Monocle3 ran on {len(monocle_sample_idx):,}/{adata.n_obs:,} cells using sparse raw counts, native preprocessing, sample alignment when multiple samples were present, and one early root cell per sample'


_run_monocle3_total(monocle_loaded, use_harmonized_umap=False)


In [ ]:
# Plot Monocle3 marker trends using the same four-panel layout.
monocle_marker_trends = plot_total_marker_trend_figure(
    monocle_loaded['TOTAL']['adata'],
    'total_monocle3',
    'TOTAL Monocle3',
    'total_monocle3_marker_trends.png',
)


## Method 4: Monocle with Harmonized UMAP


In [ ]:
# Run Monocle3 as a fourth method by learning the trajectory graph on the existing harmonized UMAP.
monocle_harmony_umap_loaded = clone_preprocessed_continua()
for obj in monocle_harmony_umap_loaded.values():
    obj['gene_lookup'] = gene_lookup

_run_monocle3_total(monocle_harmony_umap_loaded, use_harmonized_umap=True)

method_runs = {
    'diffusion_pseudotime': dpt_loaded,
    'slingshot': slingshot_loaded,
    'monocle': monocle_loaded,
    'monocle_harmony_umap': monocle_harmony_umap_loaded,
}

display(pd.DataFrame({'method_run': list(method_runs.keys())}))


In [ ]:
# Plot Monocle3-on-harmonized-UMAP marker trends using the same four-panel layout.
monocle_harmony_umap_marker_trends = plot_total_marker_trend_figure(
    monocle_harmony_umap_loaded['TOTAL']['adata'],
    'total_monocle3_harmony_umap',
    'TOTAL Monocle3 with harmonized UMAP',
    'total_monocle3_harmony_umap_marker_trends.png',
)


## Method Scoring


In [ ]:
# Score each method after its pseudotime columns have been added.
all_method_summaries = []
all_marker_summaries = []

for method_key, loaded in method_runs.items():
    for continuum, obj in loaded.items():
        print('\n' + '=' * 80)
        print(f'Scoring {method_key} for {continuum}')
        adata = obj['adata']
        config = obj['config']
        prefix = config['prefix']
        marker_axis = adata.obs[f'{prefix}_marker_axis'].to_numpy(dtype=float)
        saved_ps = adata.obs[f'{prefix}_pseudospace'].to_numpy(dtype=float)
        expected_markers = obj['expected_markers']

        method_rows = []
        marker_rows = []
        for method in obj['method_cols']:
            values = adata.obs[method].to_numpy(dtype=float)
            valid_marker = np.isfinite(values) & np.isfinite(marker_axis)
            valid_saved = np.isfinite(values) & np.isfinite(saved_ps)

            gene_rows = []
            for row in expected_markers.to_dict('records'):
                gene = row['gene']
                if gene not in adata.var_names:
                    continue
                expr_col = adata.X[:, adata.var_names.get_loc(gene)]
                expr = expr_col.toarray().ravel() if sparse.issparse(expr_col) else np.asarray(expr_col).ravel()
                expr = expr.astype(float)
                valid_expr = np.isfinite(values) & np.isfinite(expr)
                corr = spearmanr(values[valid_expr], expr[valid_expr]).correlation if valid_expr.sum() >= 20 else np.nan
                gene_rows.append({
                    'method_family': method_key,
                    'continuum': continuum,
                    'method': method,
                    'gene': gene,
                    'role': row['role'],
                    'expected_direction': row['expected_direction'],
                    'spearman_expr_vs_pseudotime': corr,
                    'matches_expected': bool(np.isfinite(corr) and np.sign(corr) == row['expected_sign']),
                })

            gene_scores = pd.DataFrame(gene_rows)
            if not gene_scores.empty:
                marker_rows.append(gene_scores)

            method_rows.append({
                'method_family': method_key,
                'continuum': prefix.upper(),
                'method': method,
                'n_cells_scored': int(np.isfinite(values).sum()),
                'spearman_with_marker_axis': spearmanr(values[valid_marker], marker_axis[valid_marker]).correlation if valid_marker.sum() >= 10 else np.nan,
                'spearman_with_saved_marker_pseudospace': spearmanr(values[valid_saved], saved_ps[valid_saved]).correlation if valid_saved.sum() >= 10 else np.nan,
                'marker_gene_direction_fraction': gene_scores['matches_expected'].mean() if len(gene_scores) else np.nan,
                'mean_abs_marker_gene_spearman': gene_scores['spearman_expr_vs_pseudotime'].abs().mean() if len(gene_scores) else np.nan,
                'note': obj['method_notes'].get(method, ''),
            })

        summary = pd.DataFrame(method_rows)
        marker_summary = pd.concat(marker_rows, ignore_index=True) if marker_rows else pd.DataFrame()
        display(summary.sort_values(['marker_gene_direction_fraction', 'spearman_with_marker_axis'], ascending=False))


        all_method_summaries.append(summary)
        all_marker_summaries.append(marker_summary if not marker_summary.empty else marker_summary)

method_summary = pd.concat(all_method_summaries, ignore_index=True)
marker_gene_direction_summary = pd.concat([df for df in all_marker_summaries if not df.empty], ignore_index=True)
display(method_summary.sort_values(['continuum', 'marker_gene_direction_fraction', 'spearman_with_marker_axis'], ascending=[True, False, False]))


## Visualization


### Prepare Method Visualization Inputs

Collects valid total-nephron methods, aligns their pseudotime vectors, and locates the embedding used for method-comparison plots.


In [ ]:
method_specs = [
    ('Scanpy DPT', dpt_loaded['TOTAL']['adata'], 'total_scanpy_dpt'),
    ('Slingshot', slingshot_loaded['TOTAL']['adata'], 'total_slingshot'),
    ('Monocle3', monocle_loaded['TOTAL']['adata'], 'total_monocle3'),
    ('Monocle3 harmonized UMAP', monocle_harmony_umap_loaded['TOTAL']['adata'], 'total_monocle3_harmony_umap'),
]
method_specs = [spec for spec in method_specs if spec[2] in spec[1].obs]
if len(method_specs) < 2:
    raise ValueError('Need at least two TOTAL pseudotime methods to compare')
method_labels = [label for label, _, _ in method_specs]

base_adata = dpt_loaded['TOTAL']['adata']
method_df = pd.DataFrame(index=base_adata.obs_names)
for label, adata, col in method_specs:
    values = pd.Series(adata.obs[col].to_numpy(dtype=float), index=adata.obs_names, name=label)
    method_df[label] = values.reindex(method_df.index)

marker_call_col = 'total_segment_marker_call'
if marker_call_col in base_adata.obs.columns:
    method_df['segment'] = base_adata.obs[marker_call_col].astype(str).reindex(method_df.index)
else:
    method_df['segment'] = 'all'

basis = None
for key in ['X_umap_harmony_total', 'X_umap_harmony', 'X_umap']:
    if key in base_adata.obsm:
        basis = key
        break
if basis is None:
    for key in base_adata.obsm.keys():
        if str(key).startswith('X_umap'):
            basis = str(key)
            break
if basis is None:
    preferred_reps = ['X_pca_harmony_total', 'X_pca_harmony', 'X_harmony', 'X_pca']
    use_rep = next((rep for rep in preferred_reps if rep in base_adata.obsm), None)
    if use_rep is None:
        sc.pp.pca(base_adata, n_comps=min(N_PCS_FALLBACK, base_adata.n_vars - 1), random_state=RANDOM_STATE)
        use_rep = 'X_pca'
    sc.pp.neighbors(base_adata, n_neighbors=N_NEIGHBORS, use_rep=use_rep, key_added='method_compare_neighbors', random_state=RANDOM_STATE)
    sc.tl.umap(base_adata, neighbors_key='method_compare_neighbors', random_state=RANDOM_STATE)
    basis = 'X_umap'

embedding = pd.DataFrame(base_adata.obsm[basis][:, :2], index=base_adata.obs_names, columns=['x', 'y'])
plot_df = embedding.join(method_df)
display(pd.DataFrame({'method': method_labels, 'pseudotime_column': [col for _, _, col in method_specs]}))

### Total Method UMAP Comparison

Plots one UMAP comparison figure, colored by each method’s total-nephron pseudospace.


In [ ]:
fig, axes = plt.subplots(1, len(method_specs), figsize=(4.2 * len(method_specs), 4.2), constrained_layout=True)
axes = np.atleast_1d(axes)
for ax, (label, _, _) in zip(axes, method_specs):
    valid = plot_df[label].notna()
    ax.scatter(plot_df.loc[~valid, 'x'], plot_df.loc[~valid, 'y'], s=3, color='lightgray', alpha=0.35, linewidths=0)
    sca = ax.scatter(
        plot_df.loc[valid, 'x'],
        plot_df.loc[valid, 'y'],
        c=plot_df.loc[valid, label],
        cmap='viridis',
        vmin=0,
        vmax=1,
        s=4 if base_adata.n_obs >= 5000 else 9,
        alpha=0.9,
        linewidths=0,
    )
    ax.set_title(f'{label}\n{valid.sum():,} scored')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel(basis.replace('X_', ''))
    ax.set_ylabel('')
fig.colorbar(sca, ax=axes, label='Normalized pseudospace', fraction=0.025, pad=0.02)
fig.suptitle('TOTAL nephron pseudospace by method', y=1.03, fontsize=14)
fig.savefig(OUTPUT_DIR / 'total_method_pseudospace_umap_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

### Method Pairwise Agreement Table

Calculates pairwise Spearman correlations between method pseudotime coordinates on shared cells.


In [ ]:
pair_rows = []
for i, x_label in enumerate(method_labels):
    for y_label in method_labels[i + 1:]:
        valid = method_df[[x_label, y_label]].notna().all(axis=1)
        corr = spearmanr(method_df.loc[valid, x_label], method_df.loc[valid, y_label]).correlation if valid.sum() >= 20 else np.nan
        pair_rows.append({'x': x_label, 'y': y_label, 'n': int(valid.sum()), 'spearman': corr})
pair_summary = pd.DataFrame(pair_rows)
display(pair_summary)

### Method Pairwise Agreement Scatter

Plots one pairwise agreement figure for method pseudotime coordinates.


In [ ]:
fig, axes = plt.subplots(1, len(pair_rows), figsize=(4.4 * len(pair_rows), 4.2), constrained_layout=True)
axes = np.atleast_1d(axes)
for ax, row in zip(axes, pair_rows):
    valid = method_df[[row['x'], row['y']]].notna().all(axis=1)
    sample = method_df.loc[valid, [row['x'], row['y'], 'segment']].dropna().copy()
    if len(sample) > 8000:
        sample = sample.sample(8000, random_state=RANDOM_STATE)
    sns.scatterplot(
        data=sample,
        x=row['x'],
        y=row['y'],
        hue='segment' if sample['segment'].nunique() > 1 else None,
        s=9,
        alpha=0.45,
        linewidth=0,
        ax=ax,
        legend=False,
    )
    ax.plot([0, 1], [0, 1], color='black', linewidth=1, alpha=0.4)
    ax.set_xlim(-0.03, 1.03)
    ax.set_ylim(-0.03, 1.03)
    ax.set_title(f"{row['x']} vs {row['y']}\nSpearman={row['spearman']:.2f}, n={row['n']:,}")
    ax.set_xlabel(row['x'])
    ax.set_ylabel(row['y'])
fig.suptitle('Pairwise TOTAL pseudospace agreement', y=1.03, fontsize=14)
fig.savefig(OUTPUT_DIR / 'total_method_pairwise_agreement.png', dpi=200, bbox_inches='tight')
plt.show()

## Save Pseudotime Outputs


In [ ]:
# Save compact AnnData objects for each TOTAL pseudotime method.
def save_total_pseudotime_anndata(source_adata, pseudotime_col, output_path):
    if pseudotime_col not in source_adata.obs.columns:
        raise KeyError(f'Missing required pseudotime column: {pseudotime_col}')

    out = adata_all_expression_filtered.copy()
    shared = out.obs_names.intersection(source_adata.obs_names)
    if pseudotime_col not in out.obs.columns:
        out.obs[pseudotime_col] = np.nan
    out.obs.loc[shared, pseudotime_col] = source_adata.obs.loc[shared, pseudotime_col].to_numpy(dtype=float)

    segment_col = 'total_segment_marker_call'
    if segment_col in source_adata.obs.columns:
        if segment_col not in out.obs.columns:
            out.obs[segment_col] = pd.NA
        out.obs.loc[shared, segment_col] = source_adata.obs.loc[shared, segment_col].astype(str).values

    if 'broad_tubule_marker_call' in adata_all.obs.columns:
        if 'broad_tubule_marker_call' not in out.obs.columns:
            out.obs['broad_tubule_marker_call'] = pd.NA
        out.obs.loc[shared, 'broad_tubule_marker_call'] = adata_all.obs.loc[shared, 'broad_tubule_marker_call'].astype(str).values

    keep_obs = [
        col for col in [
            'sample',
            'species',
            'condition',
            'broad_tubule_marker_call',
            'total_segment_marker_call',
            pseudotime_col,
        ]
        if col in out.obs.columns
    ]
    out.obs = out.obs[keep_obs].copy()
    out.write(output_path)
    print(f'Saved {pseudotime_col} AnnData to: {output_path.relative_to(PROJECT_DIR)}')
    return out


saved_pseudotime_anndatas = {
    'scanpy_dpt': save_total_pseudotime_anndata(
        dpt_loaded['TOTAL']['adata'],
        'total_scanpy_dpt',
        OUTPUT_DIR / 'all_tubules_scanpy_dpt_caleb.h5ad',
    ),
    # 'slingshot': save_total_pseudotime_anndata(
    #     slingshot_loaded['TOTAL']['adata'],
    #     'total_slingshot',
    #     OUTPUT_DIR / 'all_tubules_slingshot.h5ad',
    # ),
    # 'monocle3': save_total_pseudotime_anndata(
    #     monocle_loaded['TOTAL']['adata'],
    #     'total_monocle3',
    #     OUTPUT_DIR / 'all_tubules_monocle3.h5ad',
    # ),
    # 'monocle3_harmony_umap': save_total_pseudotime_anndata(
    #     monocle_harmony_umap_loaded['TOTAL']['adata'],
    #     'total_monocle3_harmony_umap',
    #     OUTPUT_DIR / 'all_tubules_monocle3_harmony_umap.h5ad',
    # ),
}

# Keep the root-level DPT file used by downstream notebooks in sync with the same broad labels.
ROOT_SCANPY_DPT_PATH = DATA_DIR / 'all_tubules_scanpy_dpt_caleb.h5ad'
saved_pseudotime_anndatas['scanpy_dpt'].write(ROOT_SCANPY_DPT_PATH)
print(f'Saved downstream label-compatible Scanpy DPT AnnData to: {ROOT_SCANPY_DPT_PATH.relative_to(PROJECT_DIR)}')

saved_pseudotime_anndatas
